# Ipoh ageing landscape — end-to-end measurement notebook (real data only)
Every number here comes from a named public source. Anything that can't be measured from public data is listed in **Part 8**, not filled in.

| Part | Domain | What it measures |
|---|---|---|
| 1 | People | Where people and residents aged 65+ live (Census 2020 % aged 65+, OpenDOSM growth, Meta 30 m layer, OSM boundary), on 400 m hexagons |
| 2 | **Health** | Walking and driving time to public clinics and hospitals, public-clinic capacity, hospital beds per 10,000 |
| 3 | **Money** | Official DUN household income, income inside each DUN via NAPIC house prices, poverty rate, Meta wealth cross-check |
| 4 | **Care** | Walking and driving time to care homes and senior activity centres (OSM + JKM PAWE list) |
| 5 | **Social** | Kinds of meeting place within a 10-minute walk |
| 6 | **Access** | Bus stops within 400 m, bus frequency (BAS.MY timetable), flats share (NAPIC), internet speed (Ookla) |
| 7 | Index | 0–100 domain scores, overall index, Monte Carlo weight test, DUN table, priority list |
| 8 | Gaps | What only a survey or a data request can give |

**Files to put in place before running** (all other data is downloaded and cached in `data/`):
* `mys_general_2020_csv.zip` (or the 60+ file): Meta population layer, from https://data.humdata.org/dataset/f165a732-7125-4321-baad-1b7005ed93d0
* `data/napic/*.csv|xlsx`: NAPIC Open Transaction Data, Perak → Kinta → Residential, from https://napic2.jpph.gov.my. *Optional*: without it, Money uses official DUN income only.

Needs: `pandas numpy pyarrow plotly matplotlib scipy requests openpyxl`. The first run downloads the OSM street networks and Ookla tiles, so it takes a while; later runs use the cache.

In [ ]:
import time, json, requests
OVERPASS_SERVERS = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter",
                    "https://maps.mail.ru/osm/tools/overpass/api/interpreter", "https://overpass.private.coffee/api/interpreter"]
UA = {"User-Agent": "ipoh-ageing-study/1.0 (research)"}
def overpass_json(q, rounds=3):
    for k in range(rounds):
        for url in OVERPASS_SERVERS:
            try:
                r = requests.post(url, data={"data": q}, timeout=600, headers=UA)
                if r.status_code == 200 and r.text.lstrip().startswith("{"):
                    js = r.json()
                    if js.get("elements") or "remark" not in js: return js
                    print(f"   {url.split('/')[2]}: {js['remark'][:120]}")
                else:
                    print(f"   {url.split('/')[2]} busy (HTTP {r.status_code})")
            except Exception as ex:
                print(f"   {url.split('/')[2]} failed: {str(ex)[:100]}")
        print(f"   all OpenStreetMap servers busy — waiting {30 * (k + 1)} s ({k + 1}/{rounds})"); time.sleep(30 * (k + 1))
    raise RuntimeError("OpenStreetMap (Overpass) download failed on every server. Re-run this cell in a few minutes.")
def get_json(url, what, tries=3, **kw):
    """GET that must return JSON; retries, then stops with a message naming the download."""
    kw.setdefault("timeout", 60); kw.setdefault("headers", UA); last = ""
    for k in range(tries):
        try:
            r = requests.get(url, **kw)
            if r.status_code == 200 and r.text.lstrip()[:1] in "{[": return r.json()
            last = f"HTTP {r.status_code}: {r.text[:120]!r}"
        except Exception as ex:
            last = str(ex)[:150]
        time.sleep(3 * (k + 1))
    raise RuntimeError(f"{what}: no JSON from {url} → {last}")
def nominatim(q, **params):
    """Address search; returns [] (never crashes) if the service refuses (it limits shared cloud IPs such as Colab)."""
    try:
        r = requests.get("https://nominatim.openstreetmap.org/search", params={"q": q, "format": "json", "limit": 1, "countrycodes": "my", **params},
                         headers=UA, timeout=30)
        time.sleep(1.1)
        return r.json() if r.status_code == 200 and r.text.lstrip().startswith("[") else []
    except Exception:
        return []

# PART 1 · PEOPLE — where people and older residents live

**What this notebook does**
1. Takes the **Census 2020 share aged 65+** for each census area that makes up Ipoh (DOSM, *Kinta: Fakta dan Angka*, page 8 "Profil demografi mengikut mukim").
2. Takes the **latest total population**: Census 2020 total for those areas, grown by Kinta district's real growth from 2020 to the latest OpenDOSM year.
3. Spreads people across Ipoh with **Meta's 30 m settlement-based population layer** (where buildings are) inside the **OpenStreetMap "Majlis Bandaraya Ipoh" boundary**, on 400 m hexagons.
4. Assumes each census area's **65+ share is the same today as in 2020** (as agreed).

**Be honest about one limit:** Mukim Hulu Kinta holds ~91% of Ipoh's people and has a single figure (9.2%). So the 65+ *share* can only differ between hexagons where a separate "Bandar" census area applies — and only if you give the notebook those areas' boundaries (Section 5). Without them, every hexagon gets the combined Ipoh share and the 65+ *count* map simply follows the population map. Nothing here is simulated.

Needs: `pandas numpy pyarrow plotly matplotlib requests`. Files: the Meta population CSV/ZIP and (optional) the notebook's OSM cache `data/osm_ipoh.json`.

In [ ]:
import os, io, json, math, zipfile
import numpy as np, pandas as pd
import plotly.graph_objects as go
from matplotlib.path import Path
from IPython.display import display

# ---- inputs (edit paths) -------------------------------------------------------------------------
META_FILE   = "mys_general_2020_csv.zip"            # Meta total population (preferred). Fallback below.
META_FALLBACK = "mys_elderly_60_plus_2020_csv.zip"  # inside Ipoh this layer = total × one constant, so its PATTERN equals the population pattern
OSM_CACHE   = "data/osm_ipoh.json"                  # from the main notebook; otherwise the boundary is downloaded
OPENDOSM_DISTRICT = "https://storage.dosm.gov.my/population/population_district.parquet"   # or a downloaded CSV/parquet path
BANDAR_GEOJSON = "data/census_bandar_boundaries.geojson"   # OPTIONAL: official polygons of the Bandar census areas (property "name")
ESTIMATE_BANDAR = True                              # no official polygons → estimate them (Section 5b), clearly labelled as estimates
BBOX = (4.48, 100.98, 4.82, 101.32)                 # south, west, north, east
HEX_R = 400.0

## 1 · Census 2020 figures (DOSM, *Kinta: Fakta dan Angka*, p. 8)
Typed exactly from the table. Bandar areas are counted **separately** from the mukim (the table's areas add up to Kinta's total of 888,766).
`in_ipoh` marks the areas you chose. The commented rows are other small towns that may also lie inside the City Council boundary — check them against the boundary map before switching them on.

In [ ]:
CENSUS = pd.DataFrame([
    # area,                 total,   pct_65plus, in_ipoh
    ("Mukim Hulu Kinta",    673_771,  9.2, True),
    ("Bandar Ipoh (S)",      35_577, 16.3, True),
    ("Bandar Ipoh (U)",      26_979, 10.9, True),
    ("Bandar Menglembu",        270, 19.6, True),
    ("Bandar Tambun",           913,  7.0, True),
    ("Bandar Chemor",         1_829, 10.8, False),
    ("Bandar Tanjong Rambutan", 1_311, 14.2, False),
    ("Bandar Jelapang",          22,  0.0, False),
    ("Bandar Lahat",            293,  8.9, False),
    ("Pekan Simpang Pulai",      67, 17.9, False),
], columns=["area", "pop_2020", "pct65", "in_ipoh"])
CENSUS["p65_2020"] = (CENSUS.pop_2020 * CENSUS.pct65 / 100).round()
C = CENSUS[CENSUS.in_ipoh].copy()
IPOH_POP_2020 = C.pop_2020.sum(); IPOH_P65_2020 = C.p65_2020.sum(); IPOH_PCT65 = 100 * IPOH_P65_2020 / IPOH_POP_2020
print(f"Ipoh census areas, 2020: {IPOH_POP_2020:,} people · {IPOH_P65_2020:,.0f} aged 65+ · combined share {IPOH_PCT65:.2f}%")
display(C[["area", "pop_2020", "pct65", "p65_2020"]])

Ipoh census areas, 2020: 737,510 people · 70,844 aged 65+ · combined share 9.61%


,area,pop_2020,pct65,p65_2020
0,Mukim Hulu Kinta,673771,9.2,61987.0
1,Bandar Ipoh (S),35577,16.3,5799.0
2,Bandar Ipoh (U),26979,10.9,2941.0
3,Bandar Menglembu,270,19.6,53.0
4,Bandar Tambun,913,7.0,64.0


## 2 · Latest total population
Ipoh's 2020 census total grown at **Kinta district's real growth rate** (OpenDOSM `population_district`, 2020 → latest year). Shares aged 65+ are held at their 2020 values.

In [ ]:
src = OPENDOSM_DISTRICT
d = pd.read_parquet(src) if src.endswith(".parquet") else pd.read_csv(src)
d["date"] = pd.to_datetime(d["date"])
kin = d[(d.district.str.lower() == "kinta") & (d.sex == "both") & (d.ethnicity == "overall") & (d.age == "overall")]
k = kin.set_index(kin.date.dt.year)["population"].sort_index()
print(k)                                   # all Kinta years available, in thousands
LATEST_YEAR = int(k.index.max()); GROWTH = float(k[LATEST_YEAR] / k[2020])
IPOH_POP = IPOH_POP_2020 * GROWTH
C["pop_latest"] = C.pop_2020 * GROWTH; C["p65_latest"] = C.pop_latest * C.pct65 / 100
print(f"Kinta (OpenDOSM): {k[2020]:,.1f}k in 2020 → {k[LATEST_YEAR]:,.1f}k in {LATEST_YEAR} (growth ×{GROWTH:.4f})")
print(f"Ipoh {LATEST_YEAR}: {IPOH_POP:,.0f} people · {C.p65_latest.sum():,.0f} aged 65+ (2020 shares held constant)")

date
2020    888.8
2021    893.6
2022    895.8
2023    909.1
2024    923.0
2025    927.1
2026    931.0
Name: population, dtype: float64
Kinta (OpenDOSM): 888.8k in 2020 → 931.0k in 2026 (growth ×1.0475)
Ipoh 2026: 772,527 people · 74,207 aged 65+ (2020 shares held constant)


## 3 · Ipoh boundary (OpenStreetMap)

In [ ]:
def stitch(segments):
    segs = [list(s) for s in segments if len(s) >= 2]; rings = []
    while segs:
        ring = segs.pop(0); changed = True
        while ring[0] != ring[-1] and changed:
            changed = False
            for i, s in enumerate(segs):
                if s[0] == ring[-1]: ring += s[1:]
                elif s[-1] == ring[-1]: ring += s[::-1][1:]
                elif s[-1] == ring[0]: ring = s[:-1] + ring
                elif s[0] == ring[0]: ring = s[::-1][:-1] + ring
                else: continue
                segs.pop(i); changed = True; break
        if len(ring) >= 4 and ring[0] == ring[-1]: rings.append(np.array(ring, float))
    return rings

if os.path.exists(OSM_CACHE):
    js = json.load(open(OSM_CACHE, encoding="utf-8"))
else:
    import requests
    bb = f"{BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]}"
    q = (f'[out:json][timeout:180];(relation["boundary"="administrative"]["name"~"Ipoh",i]({bb});'
         f'node["place"~"^(city|town|suburb|quarter|village|neighbourhood)$"]({bb}););out geom;')
    js = overpass_json(q)
    os.makedirs(os.path.dirname(OSM_CACHE), exist_ok=True); json.dump(js, open(OSM_CACHE, "w", encoding="utf-8"))
    print("Downloaded boundary + place points from OpenStreetMap →", OSM_CACHE)
RINGS = []
for el in js.get("elements", []):
    t = el.get("tags", {})
    if el.get("type") == "relation" and t.get("boundary") == "administrative" and "ipoh" in str(t.get("name", "")).lower():
        RINGS += stitch([[(p["lat"], p["lon"]) for p in m["geometry"]] for m in el.get("members", []) if m.get("role") in ("outer", "") and m.get("geometry")])
        print("Boundary:", t.get("name"))
assert RINGS, "No 'Ipoh' administrative boundary found in the OSM data."

Downloaded boundary + place points from OpenStreetMap → data/osm_ipoh.json
Boundary: Majlis Bandaraya Ipoh


## 4 · Spread people over 400 m hexagons (Meta 30 m layer)

In [ ]:
def read_meta(path):
    f = zipfile.ZipFile(path).open([n for n in zipfile.ZipFile(path).namelist() if n.endswith(".csv")][0]) if path.endswith(".zip") else open(path)
    parts = []
    for ch in pd.read_csv(f, chunksize=2_000_000):
        ch.columns = ["lon", "lat", "v"]
        m = ch.lat.between(BBOX[0], BBOX[2]) & ch.lon.between(BBOX[1], BBOX[3])
        if m.any(): parts.append(ch[m])
    return pd.concat(parts, ignore_index=True)
META_USED = META_FILE if os.path.exists(META_FILE) else META_FALLBACK
P = read_meta(META_USED)
inside = np.zeros(len(P), bool)
for ring in RINGS: inside |= Path(ring).contains_points(P[["lat", "lon"]].values)
P = P[inside].reset_index(drop=True)
print(f"Meta layer used: {META_USED} · {len(P):,} settled 30 m pixels inside the boundary")

LAT0, LON0 = 4.60, 101.09
KX, KY = 111_320 * math.cos(math.radians(LAT0)), 110_574
def to_xy(lat, lon): return (np.asarray(lon) - LON0) * KX, (np.asarray(lat) - LAT0) * KY
def to_ll(x, y): return LAT0 + np.asarray(y) / KY, LON0 + np.asarray(x) / KX
def pixel_to_hex(x, y, R=HEX_R):
    q = (math.sqrt(3) / 3 * x - y / 3) / R; r = (2 / 3 * y) / R; s = -q - r
    rq, rr, rs = np.round(q), np.round(r), np.round(s)
    dq, dr, ds = np.abs(rq - q), np.abs(rr - r), np.abs(rs - s)
    fq = (dq > dr) & (dq > ds); fr = ~fq & (dr > ds)
    rq = np.where(fq, -rr - rs, rq); rr = np.where(fr, -rq - rs, rr)
    return rq.astype(int), rr.astype(int)
def hex_ring(q, r, R=HEX_R):
    cx, cy = math.sqrt(3) * R * (q + r / 2), 1.5 * R * r
    a = np.deg2rad(np.arange(30, 390, 60)); la, lo = to_ll(cx + R * np.cos(a), cy + R * np.sin(a))
    ring = [[round(float(b), 6), round(float(c), 6)] for c, b in zip(la, lo)]; return ring + [ring[0]]

x, y = to_xy(P.lat, P.lon); P["q"], P["r"] = pixel_to_hex(x, y)
H = P.groupby(["q", "r"]).agg(w=("v", "sum"), lat=("lat", "mean"), lon=("lon", "mean")).reset_index()
H["hex_id"] = [f"H{q:+04d}{r:+04d}" for q, r in zip(H.q, H.r)]
H["pop"] = IPOH_POP * H.w / H.w.sum()
GEO = {"type": "FeatureCollection", "features": [{"type": "Feature", "id": h, "properties": {"id": h},
       "geometry": {"type": "Polygon", "coordinates": [hex_ring(q, r)]}} for h, q, r in zip(H.hex_id, H.q, H.r)]}
print(f"{len(H):,} hexagons · population {H['pop'].sum():,.0f}")

Meta layer used: mys_general_2020_csv.zip · 127,465 settled 30 m pixels inside the boundary
841 hexagons · population 772,527


## 5 · 65+ share per hexagon
* **With** `BANDAR_GEOJSON` (polygons of the Bandar census areas, e.g. digitised from DOSM's census maps or supplied by PLANMalaysia/MBI): hexagons inside a Bandar area get its share and its census population total; the rest get Mukim Hulu Kinta's 9.2% and total.
* **Without** it and `ESTIMATE_BANDAR = True`: the Bandar areas are **estimated** (Section 5b) — grown from their OpenStreetMap town centre until they hold their real census population.
* **Without** either: every hexagon gets the combined Ipoh share.

In [ ]:
H["census_area"] = "Ipoh (combined)"; H["pct65"] = IPOH_PCT65; P65_DONE = False
if os.path.exists(BANDAR_GEOJSON):
    bg = json.load(open(BANDAR_GEOJSON, encoding="utf-8"))
    H["census_area"] = "Mukim Hulu Kinta"; H["pct65"] = float(C.set_index("area").loc["Mukim Hulu Kinta", "pct65"])
    pts = H[["lon", "lat"]].values
    for f in bg["features"]:
        nm = f["properties"].get("name"); row = C[C.area.str.lower() == str(nm).lower()]
        if row.empty: print("  skipped (not an in_ipoh census area):", nm); continue
        polys = [f["geometry"]["coordinates"]] if f["geometry"]["type"] == "Polygon" else f["geometry"]["coordinates"]
        hit = np.zeros(len(H), bool)
        for poly in polys: hit |= Path(np.array(poly[0])[:, :2]).contains_points(pts)
        H.loc[hit, "census_area"] = row.area.iloc[0]; H.loc[hit, "pct65"] = float(row.pct65.iloc[0])
    # each census area also gets its own real census population total (latest), spread by the Meta weights inside it
    tot = C.set_index("area").pop_latest
    H["pop"] = H.groupby("census_area").w.transform(lambda s: s / s.sum()) * H.census_area.map(tot)
    print("Population and 65+ share by census area (from the boundaries you supplied):")
    print(H.groupby("census_area").agg(hexagons=("hex_id", "size"), pop=("pop", "sum")).round(0))
elif ESTIMATE_BANDAR:
    # ---- 5b · ESTIMATED Bandar areas (not official boundaries) ----------------------------------------------
    # Each Bandar area = the hexagons nearest its OSM town centre, taken in order of distance until they hold the
    # area's real census population (latest). Bandar Ipoh (U) and (S) are merged at their combined real share,
    # so no guess is needed about which half is which. A hexagon only partly filled gets a population-weighted mix.
    places = [el for el in js.get("elements", []) if el.get("type") == "node" and el.get("tags", {}).get("place") and el.get("tags", {}).get("name")]
    def osm_centre(names):
        for want in names:
            for el in places:
                if el["tags"]["name"].strip().lower() == want.lower(): return el["lat"], el["lon"], el["tags"]["name"]
        return None
    ci = C.set_index("area")
    ipoh_pop = ci.loc[["Bandar Ipoh (S)", "Bandar Ipoh (U)"], "pop_latest"].sum()
    ipoh_pct = 100 * ci.loc[["Bandar Ipoh (S)", "Bandar Ipoh (U)"], "p65_latest"].sum() / ipoh_pop
    UNITS = [("Bandar Ipoh (U+S) — estimated", ["Ipoh"], ipoh_pop, ipoh_pct)]
    for nm, look in [("Bandar Menglembu", ["Menglembu"]), ("Bandar Tambun", ["Tambun"]), ("Bandar Chemor", ["Chemor"]),
                     ("Bandar Tanjong Rambutan", ["Tanjung Rambutan", "Tanjong Rambutan"]), ("Bandar Lahat", ["Lahat"]),
                     ("Bandar Jelapang", ["Jelapang"]), ("Pekan Simpang Pulai", ["Simpang Pulai"])]:
        if nm in ci.index: UNITS.append((nm.replace("Pekan ", "Pekan ") + " — estimated", look, ci.loc[nm, "pop_latest"], ci.loc[nm, "pct65"]))
    hk_pct = float(ci.loc["Mukim Hulu Kinta", "pct65"])
    remaining = H["pop"].values.copy(); hx, hy = to_xy(H.lat.values, H.lon.values)
    take = {}
    for nm, look, target, pct in UNITS:
        c = osm_centre(look)
        if c is None:
            print(f"  ⚠ no OSM place point for {look[0]} → {nm} left inside Mukim Hulu Kinta"); continue
        cx, cy = to_xy(c[0], c[1]); order = np.argsort(np.hypot(hx - cx, hy - cy))
        t = np.zeros(len(H)); need = target
        for i in order:
            got = min(remaining[i], need); t[i] = got; remaining[i] -= got; need -= got
            if need <= 1e-6: break
        take[nm] = (t, pct); print(f"  {nm}: centre '{c[2]}', {int((t > 0).sum())} hexagon(s), {t.sum():,.0f} people at {pct:.1f}% aged 65+")
    H["p65"] = remaining * hk_pct / 100 + sum(t * pct / 100 for t, pct in take.values())
    H["pct65"] = 100 * H.p65 / H["pop"]
    main_share = pd.DataFrame({nm: t for nm, (t, _) in take.items()} | {"Mukim Hulu Kinta": remaining}, index=H.index)
    H["census_area"] = main_share.idxmax(axis=1)
    P65_DONE = True
    BANDAR_NOTE = "Bandar areas ESTIMATED: grown from OSM town centres to their real census population (not official boundaries)"
else:
    print(f"No Bandar boundaries supplied → every hexagon uses the combined Ipoh share, {IPOH_PCT65:.2f}%.")
if not P65_DONE:
    H["p65"] = H["pop"] * H.pct65 / 100
print(f"Total aged 65+: {H.p65.sum():,.0f} ({100*H.p65.sum()/H['pop'].sum():.2f}%)")

  Bandar Ipoh (U+S) — estimated: centre 'Ipoh', 39 hexagon(s), 65,526 people at 14.0% aged 65+
  Bandar Menglembu — estimated: centre 'Menglembu', 1 hexagon(s), 283 people at 19.6% aged 65+
  Bandar Tambun — estimated: centre 'Tambun', 1 hexagon(s), 956 people at 7.0% aged 65+
Total aged 65+: 74,207 (9.61%)


## 6 · Maps — white background, no gridlines

In [ ]:
FONT = "Inter, system-ui, sans-serif"
BLUES = [[0, "#eef4fc"], [0.3, "#cde2fb"], [0.55, "#86b6ef"], [0.8, "#1c5cab"], [1, "#0d366b"]]
ORANGES = [[0, "#fff4ec"], [0.3, "#fde0cc"], [0.55, "#f8b48c"], [0.8, "#b9461a"], [1, "#6e2408"]]
CENTER = dict(lat=float(H.lat.mean()), lon=float(H.lon.mean()))
NOTE = (f"Population: Census 2020 grown to {LATEST_YEAR} with Kinta's growth (OpenDOSM), spread by Meta's 30 m layer · "
        "65+ share: Census 2020 (DOSM) held constant · boundary © OpenStreetMap contributors")
def hexmap(col, cs, title, sub, fmt, zmax=None, label=""):
    fig = go.Figure(go.Choroplethmap(geojson=GEO, locations=H.hex_id, z=H[col], colorscale=cs, zmin=0, zmax=zmax,
                    marker=dict(opacity=0.85, line=dict(width=0.3, color="white")),
                    colorbar=dict(title=dict(text=label, side="top"), thickness=10, len=0.45, x=0.01, xanchor="left", y=0.97, yanchor="top", outlinewidth=0, bgcolor="rgba(255,255,255,0.85)"),
                    customdata=np.stack([H.census_area, H["pop"].round(0), H.p65.round(0)], -1),
                    hovertemplate="%{customdata[0]}<br>People: %{customdata[1]:,.0f}<br>Aged 65+: %{customdata[2]:,.0f}<br>" + label + ": %{z:" + fmt + "}<extra></extra>"))
    for ring in RINGS:
        fig.add_trace(go.Scattermap(lat=ring[:, 0], lon=ring[:, 1], mode="lines", line=dict(width=2, color="#14171C"), hoverinfo="skip", showlegend=False))
    fig.update_layout(map=dict(style="carto-positron", center=CENTER, zoom=10), height=720, paper_bgcolor="white", font=dict(family=FONT, color="#52514e"),
                      margin=dict(t=95, b=55, l=10, r=10),
                      title=dict(text=f"<b>{title}</b><br><span style='font-size:13px;color:#52514e'>{sub}</span>", x=0.01, xanchor="left"))
    fig.add_annotation(text=NOTE, x=0, xref="paper", y=0, yref="paper", yanchor="top", yshift=-12, showarrow=False, font=dict(size=10.5, color="#898781"), align="left")
    return fig

hexmap("pop", BLUES, f"Where Ipoh's {IPOH_POP:,.0f} people live ({LATEST_YEAR})", "Residents per 400 m hexagon", ",.0f",
       zmax=float(H["pop"].quantile(0.98)), label="Residents").show()
hexmap("p65", ORANGES, f"Where Ipoh's {H.p65.sum():,.0f} residents aged 65+ live", "Residents aged 65+ per 400 m hexagon", ",.0f",
       zmax=float(H.p65.quantile(0.98)), label="Aged 65+").show()
hexmap("pct65", ORANGES, "Share of residents aged 65+",
       ("By census area (your Bandar boundaries)" if os.path.exists(BANDAR_GEOJSON) else (BANDAR_NOTE if ESTIMATE_BANDAR else f"One figure for all of Ipoh ({IPOH_PCT65:.1f}%) — the census gives no finer split without Bandar boundaries")),
       ".1f", zmax=max(20, float(H.pct65.max())), label="% aged 65+").show()

In [ ]:
bar = C.sort_values("pct65")
fig = go.Figure(go.Bar(x=bar.pct65, y=bar.area, orientation="h", marker=dict(color=["#eb6834" if v > IPOH_PCT65 else "#c3c2b7" for v in bar.pct65]),
                       text=[f"{v:.1f}%  ·  {p:,.0f} aged 65+" for v, p in zip(bar.pct65, bar.p65_latest)], textposition="outside",
                       hovertemplate="%{y}: %{x:.1f}%<extra></extra>"))
fig.add_shape(type="line", x0=IPOH_PCT65, x1=IPOH_PCT65, y0=-0.5, y1=len(bar) - 0.5, line=dict(color="#14171C", width=1, dash="dot"))
fig.add_annotation(text=f"Ipoh {IPOH_PCT65:.1f}%", x=IPOH_PCT65, y=len(bar) - 0.5, yanchor="bottom", showarrow=False, font=dict(size=11))
fig.update_layout(height=380, paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color="#52514e"), margin=dict(t=90, b=50, l=170, r=160),
                  title=dict(text="<b>Census 2020: share aged 65+ by census area in Ipoh</b><br><span style='font-size:13px'>DOSM, Kinta: Fakta dan Angka · counts grown to "
                             f"{LATEST_YEAR}</span>", x=0.01, xanchor="left"),
                  xaxis=dict(showgrid=False, zeroline=False, ticksuffix="%", range=[0, bar.pct65.max() * 1.5], linecolor="#c3c2b7"), yaxis=dict(showgrid=False))
fig.show()

In [ ]:
os.makedirs("outputs", exist_ok=True)
H[["hex_id", "lat", "lon", "census_area", "pop", "pct65", "p65"]].round(2).to_csv("outputs/ipoh_pop65_by_hex.csv", index=False)
print("Saved outputs/ipoh_pop65_by_hex.csv")

Saved outputs/ipoh_pop65_by_hex.csv


# PART 2 · HEALTH — how quickly can older residents reach care?
Public clinics (Klinik Kesihatan) and hospitals only: OpenStreetMap is too incomplete for private GPs and pharmacies. Walking times follow real streets at an older person's pace; driving times follow real roads (one-way streets respected, no traffic).

In [ ]:
# ==================================================================================================
# IPOH — HEALTH ACCESS FOR OLDER RESIDENTS (single cell, real data only)
#   Facilities : OpenStreetMap (Overpass) + Healthsites.io Malaysia (HDX), de-duplicated
#   Roads      : OpenStreetMap walkable road/path network → real walking distances (not straight lines)
#   People     : outputs/ipoh_pop65_by_hex.csv (from the 65+ notebook)
#   Outputs    : walking time to the nearest PUBLIC CLINIC (Klinik Kesihatan) and HOSPITAL as smooth heat maps,
#                public-clinic capacity per resident (E2SFCA), and how many residents aged 65+ live how far away.
#   Private GPs and pharmacies are left out: OpenStreetMap has only a fraction of them in Ipoh.
#   Not in any public data → NOT used: clinic staffing/opening hours, illness rates by area.
#   Nothing is simulated. If a download fails the cell stops with a message.
# ==================================================================================================
import os, re, json, math, requests
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
from IPython.display import display
from matplotlib.path import Path

HEX_CSV   = "outputs/ipoh_pop65_by_hex.csv"
OSM_HEALTH_CACHE = "data/osm_ipoh_health.json"
OSM_ROADS_CACHE  = "data/osm_ipoh_walk_network.json"
HEALTHSITES_FILE = "data/malaysia_healthsites.csv"     # downloaded from HDX if missing
OSM_CACHE = "data/osm_ipoh.json"                        # optional Ipoh outline
BBOX      = (4.45, 100.98, 4.82, 101.32)                # south, west, north, east (a margin round Ipoh so border facilities count)
WALK_MPS  = 0.9      # walking speed, m/s — a SETTING (older adults walk slower than the ~1.2–1.4 m/s adult norm); change freely
E2SFCA_M  = 3000     # catchment for clinic-capacity score (network metres)
OSM_DRIVE_CACHE = "data/osm_ipoh_drive_network.json"
# Driving: OSM "maxspeed" is used where mapped; elsewhere these typical speeds by road type (km/h) — a SETTING, no congestion.
DRIVE_KMH = {"motorway": 90, "trunk": 70, "primary": 55, "secondary": 45, "tertiary": 40, "unclassified": 30, "residential": 25,
             "living_street": 15, "service": 15, "road": 30, "motorway_link": 50, "trunk_link": 40, "primary_link": 35,
             "secondary_link": 30, "tertiary_link": 30}
ACCESS_MIN = 2.0     # minutes added per car trip (getting to the car, parking, walking in) — a SETTING
OVERPASS  = "https://overpass-api.de/api/interpreter"
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True)
LAT0, LON0 = 4.60, 101.09
KX, KY = 111_320 * math.cos(math.radians(LAT0)), 110_574
def to_xy(lat, lon): return (np.asarray(lon, float) - LON0) * KX, (np.asarray(lat, float) - LAT0) * KY
def to_ll(x, y): return LAT0 + np.asarray(y) / KY, LON0 + np.asarray(x) / KX
OVERPASS_MIRRORS = [OVERPASS, "https://overpass.kumi.systems/api/interpreter",
                    "https://maps.mail.ru/osm/tools/overpass/api/interpreter", "https://overpass.private.coffee/api/interpreter"]
def overpass(q, cache, rounds=3):
    """Same OpenStreetMap query, tried on several public Overpass servers (they are often busy: 502/504/429)."""
    import time
    if os.path.exists(cache): return json.load(open(cache, encoding="utf-8"))
    for k in range(rounds):
        for url in OVERPASS_MIRRORS:
            try:
                print(f"Downloading from OpenStreetMap ({url.split('/')[2]}) …", cache)
                r = requests.post(url, data={"data": q}, timeout=600, headers={"User-Agent": "ipoh-health-access/1.0"})
                if r.status_code == 200 and r.text.lstrip().startswith("{"):
                    js = r.json()
                    if "remark" in js and not js.get("elements"): raise RuntimeError(js["remark"][:200])
                    json.dump(js, open(cache, "w", encoding="utf-8")); return js
                print(f"   busy/failed: HTTP {r.status_code}")
            except Exception as ex:
                print(f"   failed: {str(ex)[:120]}")
        wait = 30 * (k + 1); print(f"All servers busy — waiting {wait}s and trying again ({k + 1}/{rounds})"); time.sleep(wait)
    raise RuntimeError("Every Overpass server failed. Re-run this cell in a few minutes (finished downloads are cached).")
bb = f"{BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]}"

# ---- 1. people ------------------------------------------------------------------------------------
assert os.path.exists(HEX_CSV), f"{HEX_CSV} missing — run ipoh_population_65plus.ipynb first."
H = pd.read_csv(HEX_CSV); assert {"hex_id", "lat", "lon", "pop", "p65"} <= set(H.columns)
print(f"People: {H['pop'].sum():,.0f} in {len(H):,} hexagons · aged 65+: {H.p65.sum():,.0f}")

# ---- 2. health facilities: OSM ----------------------------------------------------------------------
q = (f'[out:json][timeout:300];('
     f'nwr["amenity"~"^(hospital|clinic|doctors|pharmacy|dentist)$"]({bb});'
     f'nwr["healthcare"~"^(hospital|clinic|doctor|pharmacy|centre|dialysis|rehabilitation|dentist)$"]({bb}););out center tags;')
F = []
for el in overpass(q, OSM_HEALTH_CACHE).get("elements", []):
    t = el.get("tags", {}); la = el.get("lat", el.get("center", {}).get("lat")); lo = el.get("lon", el.get("center", {}).get("lon"))
    if la is None: continue
    F.append(dict(src="OSM", osm_id=f"{el['type'][0]}{el['id']}", lat=la, lon=lo, name=t.get("name", t.get("name:en", "")),
                  amenity=t.get("amenity", ""), healthcare=t.get("healthcare", ""), operator=t.get("operator", ""),
                  operator_type=t.get("operator:type", ""), beds=t.get("beds", "")))
F = pd.DataFrame(F); print(f"OSM: {len(F):,} health features")

# ---- 3. health facilities: Healthsites.io (HDX) ------------------------------------------------------
if not os.path.exists(HEALTHSITES_FILE):
    pk = requests.get("https://data.humdata.org/api/3/action/package_show", params={"id": "malaysia-healthsites"}, timeout=60).json()["result"]
    res = [r for r in pk["resources"] if r.get("format", "").lower() == "csv" or r.get("url", "").lower().endswith(".csv")]
    assert res, "No CSV in HDX 'malaysia-healthsites' — download it by hand from https://data.humdata.org/dataset/malaysia-healthsites"
    open(HEALTHSITES_FILE, "wb").write(requests.get(res[0]["url"], timeout=300).content); print("Healthsites:", res[0]["name"])
hs = pd.read_csv(HEALTHSITES_FILE, low_memory=False)
if str(hs.iloc[0, 0]).startswith("#"): hs = hs.iloc[1:]                       # HXL tag row
lc = {c.lower(): c for c in hs.columns}
xcol = lc.get("x") or lc.get("longitude") or lc.get("lon"); ycol = lc.get("y") or lc.get("latitude") or lc.get("lat")
hs = hs.assign(lon=pd.to_numeric(hs[xcol], errors="coerce"), lat=pd.to_numeric(hs[ycol], errors="coerce")).dropna(subset=["lat", "lon"])
hs = hs[hs.lat.between(BBOX[0], BBOX[2]) & hs.lon.between(BBOX[1], BBOX[3])]
g = lambda c: hs[lc[c]].fillna("").astype(str) if c in lc else ""
HS = pd.DataFrame(dict(src="Healthsites", osm_id=(g("osm_type").str[:1] + g("osm_id").str.replace(r"\.0$", "", regex=True)) if "osm_id" in lc else "",
                       lat=hs.lat, lon=hs.lon, name=g("name"), amenity=g("amenity"), healthcare=g("healthcare"),
                       operator=g("operator"), operator_type=g("operator_type"), beds=g("beds")))
# Healthsites largely mirrors OSM → drop rows already in OSM (same id, or same type within 40 m)
HS = HS[~HS.osm_id.isin(F.osm_id)] if len(F) else HS
print(f"Healthsites: {len(hs):,} in the area, {len(HS):,} not already in OSM by id")

# ---- 4. classify --------------------------------------------------------------------------------------
A = pd.concat([F, HS], ignore_index=True)
nm = A.name.str.lower(); op = (A.operator + " " + A.operator_type).str.lower(); kind = A.amenity.str.lower() + " " + A.healthcare.str.lower()
public_kw = r"klinik kesihatan|klinik desa|klinik komuniti|\bkk\b|1malaysia|health clinic|ibu dan anak"
gov_op = op.str.contains(r"kementerian kesihatan|ministry of health|\bkkm\b|\bmoh\b|government|public|kerajaan")
A["type"] = np.select(
    [kind.str.contains("hospital") | nm.str.contains(r"^hospital|\bhospital\b"),
     kind.str.contains("pharmacy") | nm.str.contains(r"farmasi|pharmacy"),
     kind.str.contains("dentist") | nm.str.contains(r"pergigian|dental"),
     kind.str.contains("dialysis") | nm.str.contains(r"dialisis|dialysis|haemodialysis"),
     nm.str.contains(public_kw) | (kind.str.contains("clinic|doctor") & gov_op),
     kind.str.contains("clinic|doctor|centre")],
    ["Hospital", "Pharmacy", "Dental", "Dialysis", "Public clinic (KKM)", "Private clinic / GP"], "Other")
A.loc[nm.str.contains(r"\bdarah\b|blood bank|blood centre|tabung darah"), "type"] = "Other"   # blood centres are not clinics
A["public"] = gov_op | nm.str.contains(public_kw + r"|raja permaisuri bainun|hospital ipoh")

# ---- 4b. public clinics on the official KKM list but missing from OSM → placed by their official address ----
#      Source: MOH "Fasiliti Kesihatan KKM" page (dental units inside Klinik Kesihatan, Perak table, 2024).
#      Only clinics not already found are added; each is geocoded with OpenStreetMap's Nominatim and flagged.
#      If geocoding fails, type the coordinates into MANUAL_LATLON (from Google Maps) — nothing is guessed.
KKM_OFFICIAL = {"Klinik Kesihatan Chemor": "Jalan Tanah Hitam, 31200 Chemor, Perak",
                "Klinik Kesihatan UTC Ipoh": "UTC Perak, Jalan Onn Jaafar, 30300 Ipoh, Perak"}
MANUAL_LATLON = {}   # e.g. {"Klinik Kesihatan Chemor": (4.72, 101.12)}
GEO_CACHE = "data/nominatim_kkm.json"; geo = json.load(open(GEO_CACHE)) if os.path.exists(GEO_CACHE) else {}
have = " | ".join(A.loc[A.type == "Public clinic (KKM)", "name"].str.lower())
add = []
for name, addr in KKM_OFFICIAL.items():
    key = name.lower().replace("klinik kesihatan ", "")
    if key in have: continue
    if name in MANUAL_LATLON:
        la, lo = MANUAL_LATLON[name]; how = "manual"
    else:
        if name not in geo:
            for qq in (f"{name}, {addr}", addr, f"{name}, Perak"):
                r = requests.get("https://nominatim.openstreetmap.org/search", params={"q": qq, "format": "json", "limit": 1, "countrycodes": "my"},
                                 headers={"User-Agent": "ipoh-health-access/1.0"}, timeout=30).json()
                import time; time.sleep(1.1)
                if r: geo[name] = [float(r[0]["lat"]), float(r[0]["lon"]), qq]; break
            json.dump(geo, open(GEO_CACHE, "w"))
        if name not in geo: print(f"  ⚠ {name}: address not found — add it to MANUAL_LATLON"); continue
        la, lo, how = geo[name][0], geo[name][1], f"geocoded from '{geo[name][2]}'"
    print(f"  + added from KKM list: {name} ({how})")
    add.append(dict(src="KKM list", osm_id="", lat=la, lon=lo, name=name, amenity="clinic", healthcare="clinic",
                    operator="Kementerian Kesihatan Malaysia", operator_type="government", beds="", type="Public clinic (KKM)", public=True))
if add: A = pd.concat([A, pd.DataFrame(add)], ignore_index=True)
# de-duplicate across sources: same type within 40 m → keep OSM
x, y = to_xy(A.lat, A.lon); A["x"], A["y"] = x, y; keep = np.ones(len(A), bool)
for ty, grp in A.groupby("type"):
    if len(grp) < 2: continue
    t = cKDTree(grp[["x", "y"]].values)
    for i, j in t.query_pairs(40):
        a, b = grp.index[i], grp.index[j]
        drop = b if A.at[a, "src"] == "OSM" or A.at[b, "src"] != "OSM" else a
        keep[A.index.get_loc(drop)] = False
A = A[keep & A.type.isin(["Public clinic (KKM)", "Hospital"])].reset_index(drop=True)   # only public clinics + hospitals: OSM is too incomplete for private GPs/pharmacies
print("\nFacilities used (after de-duplication):"); display(A.groupby(["type", "src"]).size().unstack(fill_value=0))
print("Public clinics used — CHECK against KKM's list for Kinta; add any missing ones to KKM_OFFICIAL:")
print("  " + "\n  ".join(sorted(A.loc[A.type == "Public clinic (KKM)", "name"].replace("", "(no name)"))))

# ---- 5. walking network (OSM) ------------------------------------------------------------------------
q = (f'[out:json][timeout:600];way["highway"~"^(trunk|trunk_link|primary|primary_link|secondary|secondary_link|tertiary|tertiary_link|'
     f'unclassified|residential|living_street|service|pedestrian|footway|path|steps|track|road)$"]["foot"!="no"]["access"!="private"]({bb});out geom;')
W = overpass(q, OSM_ROADS_CACHE)
ids, lat_, lon_, ea, eb = {}, [], [], [], []
def nid(n, la, lo):
    if n not in ids: ids[n] = len(lat_); lat_.append(la); lon_.append(lo)
    return ids[n]
for el in W.get("elements", []):
    if el.get("type") != "way" or "geometry" not in el: continue
    seq = [nid(n, p["lat"], p["lon"]) for n, p in zip(el["nodes"], el["geometry"])]
    ea += seq[:-1]; eb += seq[1:]
nx, ny = to_xy(lat_, lon_); ea, eb = np.array(ea), np.array(eb)
L = np.hypot(nx[ea] - nx[eb], ny[ea] - ny[eb])
G = coo_matrix((np.r_[L, L], (np.r_[ea, eb], np.r_[eb, ea])), shape=(len(nx),) * 2).tocsr()
ncomp, lab = connected_components(G, directed=False); main = np.bincount(lab).argmax(); onmain = np.where(lab == main)[0]
print(f"\nWalking network: {len(nx):,} nodes, {len(L):,} segments, {L.sum() / 1000:,.0f} km · main connected part {len(onmain) / len(nx):.0%} of nodes")
tree = cKDTree(np.c_[nx[onmain], ny[onmain]])
def snap(xs, ys):
    d, i = tree.query(np.c_[xs, ys]); return onmain[i], d
hx, hy = to_xy(H.lat, H.lon); H_node, H_snap = snap(hx, hy)
A["node"], A["snap_m"] = snap(A.x, A.y)
print(f"Hexagon centres snapped to the network: median {np.median(H_snap):.0f} m, 95th pct {np.percentile(H_snap, 95):.0f} m")

# ---- 6. walking distance from EVERY street point to the nearest facility ---------------------------
NN = G.shape[0]
def nearest_all(mask):
    """One Dijkstra run from a virtual start node linked to every facility (link = facility's distance to the street).
       Returns metres for every network node."""
    s = A.loc[mask].groupby("node").snap_m.min() + 1e-3
    Gc = G.tocoo()
    E = coo_matrix((np.r_[Gc.data, s.values], (np.r_[Gc.row, np.full(len(s), NN)], np.r_[Gc.col, s.index.values])), shape=(NN + 1, NN + 1)).tocsr()
    return dijkstra(E, directed=True, indices=NN, limit=40_000)[:NN]
TYPES = ["Public clinic (KKM)", "Hospital"]
NODE_MIN = {}
for ty in TYPES:
    m = nearest_all(A.type == ty)
    NODE_MIN[ty] = m / WALK_MPS / 60
    H["min_" + ty] = (m[H_node] + H_snap) / WALK_MPS / 60

# ---- 6b. DRIVING time (OSM drivable roads, one-way streets respected) ----------------------------------
q = (f'[out:json][timeout:600];way["highway"~"^({"|".join(DRIVE_KMH)})$"]["access"!~"^(private|no)$"]({bb});out geom;')
DW = overpass(q, OSM_DRIVE_CACHE)
dids, dlat, dlon, u, v, tsec = {}, [], [], [], [], []
def did(n, la, lo):
    if n not in dids: dids[n] = len(dlat); dlat.append(la); dlon.append(lo)
    return dids[n]
n_ms = 0
for el in DW.get("elements", []):
    if el.get("type") != "way" or "geometry" not in el: continue
    t = el.get("tags", {}); hw = t.get("highway", "road")
    ms = re.match(r"\s*(\d+)", str(t.get("maxspeed", "")))
    kmh = float(ms.group(1)) if ms else DRIVE_KMH.get(hw, 30); n_ms += bool(ms)
    seq = [did(n, p["lat"], p["lon"]) for n, p in zip(el["nodes"], el["geometry"])]
    ow = t.get("oneway", "no"); rev = ow == "-1"; one = ow in ("yes", "true", "1", "-1") or t.get("junction") == "roundabout" or hw == "motorway"
    for a, b in zip(seq[:-1], seq[1:]):
        if rev: a, b = b, a
        u.append(a); v.append(b); tsec.append(kmh)
        if not one: u.append(b); v.append(a); tsec.append(kmh)
dx, dy = to_xy(dlat, dlon); u, v = np.array(u), np.array(v)
tsec = np.hypot(dx[u] - dx[v], dy[u] - dy[v]) / (np.array(tsec) / 3.6) + 1e-3
GD = coo_matrix((tsec, (u, v)), shape=(len(dx),) * 2).tocsr()
_, dlab = connected_components(GD, directed=True, connection="strong"); dmain = np.where(dlab == np.bincount(dlab).argmax())[0]
print(f"\nDriving network: {len(dx):,} nodes, {len(u):,} directed segments · {n_ms:,} ways have a mapped speed limit, the rest use DRIVE_KMH")
dtree = cKDTree(np.c_[dx[dmain], dy[dmain]])
def dsnap(xs, ys):
    d, i = dtree.query(np.c_[xs, ys]); return dmain[i], d
H_dnode, H_dsnap = dsnap(hx, hy); A["dnode"], A["dsnap"] = dsnap(A.x, A.y)
GDT = GD.T.tocsr(); DN = GD.shape[0]
def drive_all(mask):
    """Minutes from every road node TO the nearest facility (reverse graph, virtual start at the facilities)."""
    s = A.loc[mask].groupby("dnode").size()
    Gc = GDT.tocoo()
    E = coo_matrix((np.r_[Gc.data, np.full(len(s), 1e-3)], (np.r_[Gc.row, np.full(len(s), DN)], np.r_[Gc.col, s.index.values])), shape=(DN + 1, DN + 1)).tocsr()
    return dijkstra(E, directed=True, indices=DN)[:DN] / 60
DNODE_MIN = {}
for ty in TYPES:
    m = drive_all(A.type == ty); DNODE_MIN[ty] = m + ACCESS_MIN
    H["drive_" + ty] = m[H_dnode] + ACCESS_MIN + H_dsnap / WALK_MPS / 60      # + walk from home to the nearest road

# ---- 7. public-clinic capacity per resident (E2SFCA, Gaussian decay on walking distance) -------------
#      Supply = 1 per clinic (no public staffing data) → "public clinics within reach per 10,000 residents".
C = A[A.type == "Public clinic (KKM)"].reset_index(drop=True)
Dall = dijkstra(G, directed=False, indices=C.node.values, limit=E2SFCA_M * 1.2) + C.snap_m.values[:, None]   # clinics × all nodes
decay = lambda d: np.where(d <= E2SFCA_M, np.exp(-0.5 * (d / (E2SFCA_M / 2)) ** 2), 0.0)
Wh = decay(Dall[:, H_node] + H_snap[None])
R = 1.0 / np.maximum((Wh * H["pop"].values[None]).sum(1), 1e-9)            # each clinic shared among everyone who can reach it
H["clinics_per_10k"] = 10_000 * (Wh * R[:, None]).sum(0)
NODE_CAP = 10_000 * (decay(Dall) * R[:, None]).sum(0); del Dall

# ---- 8. summaries ------------------------------------------------------------------------------------
BANDS = [0, 15, 30, 45, np.inf]; BL = ["< 15 min", "15–30 min", "30–45 min", "> 45 min"]
SUM = pd.DataFrame([100 * H.p65.groupby(pd.cut(H["min_" + ty], BANDS, labels=BL, right=False), observed=False).sum() / H.p65.sum() for ty in TYPES], index=TYPES)
print(f"\nShare of residents aged 65+ by WALKING time to the nearest facility (at {WALK_MPS} m/s):"); display(SUM.round(1))
DB = [0, 5, 10, 15, 20, np.inf]; DBL = ["< 5 min", "5–10 min", "10–15 min", "15–20 min", "> 20 min"]
DSUM = pd.DataFrame([100 * H.p65.groupby(pd.cut(H["drive_" + ty], DB, labels=DBL, right=False), observed=False).sum() / H.p65.sum() for ty in TYPES], index=TYPES)
print(f"Share of residents aged 65+ by DRIVING time (incl. {ACCESS_MIN:.0f} min access, no traffic):"); display(DSUM.round(1))
far = H["min_Public clinic (KKM)"] > 30
print(f"→ {H.loc[far, 'p65'].sum():,.0f} residents aged 65+ ({100 * H.loc[far, 'p65'].sum() / H.p65.sum():.0f}%) are more than a 30-minute walk from a public clinic.")

# ---- 9. charts: smooth heat surfaces ------------------------------------------------------------------
import base64, io as _io
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from scipy.interpolate import griddata
FONT = "Inter, system-ui, sans-serif"; INK = "#52514e"
TIME_CS = [[0, "#fff7e6"], [0.25, "#fdd49e"], [0.5, "#fc8d59"], [0.75, "#d7301f"], [1, "#7f0000"]]      # light = close, dark = far
CAP_CS  = [[0, "#eef4fc"], [0.3, "#cde2fb"], [0.55, "#86b6ef"], [0.8, "#1c5cab"], [1, "#0d366b"]]
HEAT_CS = [[0, "rgba(255,247,230,0)"], [0.15, "#fdd49e"], [0.45, "#fc8d59"], [0.75, "#d7301f"], [1, "#7f0000"]]
mpl_cmap = lambda cs: LinearSegmentedColormap.from_list("c", [(a, c) for a, c in cs])
CENTER = dict(lat=float(H.lat.mean()), lon=float(H.lon.mean()))
NOTE = "Facilities: © OpenStreetMap contributors + Healthsites.io + MOH address list · walking network: OpenStreetMap · people: ipoh_pop65_by_hex.csv"
def stitch(segments):
    segs = [list(map(tuple, s)) for s in segments if len(s) >= 2]; rings = []
    while segs:
        ring = segs.pop(0); changed = True
        while ring[0] != ring[-1] and changed:
            changed = False
            for i, sg in enumerate(segs):
                if sg[0] == ring[-1]: ring += sg[1:]
                elif sg[-1] == ring[-1]: ring += sg[::-1][1:]
                elif sg[-1] == ring[0]: ring = sg[:-1] + ring
                elif sg[0] == ring[0]: ring = sg[::-1][:-1] + ring
                else: continue
                segs.pop(i); changed = True; break
        if len(ring) >= 4 and ring[0] == ring[-1]: rings.append(np.array(ring, float))
    return rings
RINGS = []
if os.path.exists(OSM_CACHE):
    for el in json.load(open(OSM_CACHE, encoding="utf-8")).get("elements", []):
        t = el.get("tags", {})
        if el.get("type") == "relation" and t.get("boundary") == "administrative" and "ipoh" in str(t.get("name", "")).lower():
            for m in el.get("members", []):
                if m.get("role") in ("outer", "") and m.get("geometry"):
                    RINGS.append([(p["lat"], p["lon"]) for p in m["geometry"]])
RINGS = stitch(RINGS)

# raster grid (~80 m) over Ipoh; a pixel is shown only near a street AND inside Ipoh's populated hexagons
pad = 0.01
LON_A, LON_B = H.lon.min() - pad, H.lon.max() + pad; LAT_A, LAT_B = H.lat.min() - pad, H.lat.max() + pad
glon = np.arange(LON_A, LON_B, 0.0005); glat = np.arange(LAT_A, LAT_B, 0.0005)
GLON, GLAT = np.meshgrid(glon, glat); gx, gy = to_xy(GLAT.ravel(), GLON.ravel())
near_street = tree.query(np.c_[gx, gy], distance_upper_bound=350)[0] < 350          # blank out forest / hills / mining ponds
if RINGS:                                                                          # inside the Ipoh City Council boundary
    in_ipoh = np.zeros(len(gx), bool)
    for r in RINGS: in_ipoh |= Path(r[:, ::-1]).contains_points(np.c_[GLON.ravel(), GLAT.ravel()])
else:
    in_ipoh = cKDTree(np.c_[hx, hy]).query(np.c_[gx, gy], distance_upper_bound=800)[0] < 800
MASK = (near_street & in_ipoh).reshape(GLON.shape)
sel = np.where((nx > to_xy(0, LON_A)[0] - 500) & (nx < to_xy(0, LON_B)[0] + 500) & (ny > to_xy(LAT_A, 0)[1] - 500) & (ny < to_xy(LAT_B, 0)[1] + 500))[0]
sel = np.intersect1d(sel, onmain)
dsel = np.where((dx > to_xy(0, LON_A)[0] - 500) & (dx < to_xy(0, LON_B)[0] + 500) & (dy > to_xy(LAT_A, 0)[1] - 500) & (dy < to_xy(LAT_B, 0)[1] + 500))[0]
dsel = np.intersect1d(dsel, dmain)
def surface(node_vals, drive=False):
    ii, X, Y = (dsel, dx, dy) if drive else (sel, nx, ny)
    v = node_vals[ii]; ok = np.isfinite(v)
    Z = griddata(np.c_[X[ii][ok], Y[ii][ok]], v[ok], (gx, gy), method="linear").reshape(GLON.shape)
    Zn = griddata(np.c_[X[ii][ok], Y[ii][ok]], v[ok], (gx, gy), method="nearest").reshape(GLON.shape)
    return np.where(MASK, np.where(np.isfinite(Z), Z, Zn), np.nan)
def heat_layer(Z, cs, vmin, vmax, contours=None):
    fig_m = plt.figure(figsize=(Z.shape[1] / 100, Z.shape[0] / 100), dpi=100); ax = fig_m.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.imshow(np.clip(Z, vmin, vmax), origin="lower", extent=[LON_A, LON_B, LAT_A, LAT_B], cmap=mpl_cmap(cs), norm=Normalize(vmin, vmax),
              interpolation="bilinear", alpha=0.8, aspect="auto")
    if contours:
        cn = ax.contour(GLON, GLAT, Z, levels=contours, colors="#14171C", linewidths=0.8, alpha=0.7)
        ax.clabel(cn, fmt=lambda v: f"{v:.0f} min", fontsize=7)
    ax.set_xlim(LON_A, LON_B); ax.set_ylim(LAT_A, LAT_B)
    b = _io.BytesIO(); fig_m.savefig(b, format="png", transparent=True); plt.close(fig_m)
    return dict(sourcetype="image", source="data:image/png;base64," + base64.b64encode(b.getvalue()).decode(),
                coordinates=[[LON_A, LAT_B], [LON_B, LAT_B], [LON_B, LAT_A], [LON_A, LAT_A]], below="traces")
def legend_bar(cs, vmin, vmax, label):     # invisible trace that only draws the colour bar
    return go.Scattermap(lat=[CENTER["lat"]], lon=[CENTER["lon"]], mode="markers", hoverinfo="skip", showlegend=False,
                         marker=dict(size=0.1, color=[vmin], cmin=vmin, cmax=vmax, colorscale=cs, showscale=True,
                                     colorbar=dict(title=dict(text=label, side="top"), thickness=10, len=0.4, x=0.01, xanchor="left", y=0.97, yanchor="top", outlinewidth=0)))
def facility_traces(types=TYPES):
    out = []
    for ty, col, sym, sz in [("Hospital", "#14171C", "hospital", 14), ("Public clinic (KKM)", "#1c5cab", "doctor", 12)]:
        if ty not in types: continue
        s = A[A.type == ty]
        out.append(go.Scattermap(lat=s.lat, lon=s.lon, mode="markers", name=f"{ty} ({len(s)})", marker=dict(size=sz, color=col),
                                 text=s.name.replace("", "(no name)"), hovertemplate="%{text}<extra>" + ty + "</extra>"))
    return out
def map_fig(traces, title, sub, layers=(), note=None):
    fig = go.Figure(traces)
    for r in RINGS:
        fig.add_trace(go.Scattermap(lat=r[:, 0], lon=r[:, 1], mode="lines", line=dict(width=1.5, color="#14171C"), hoverinfo="skip", showlegend=False))
    fig.update_layout(map=dict(style="carto-positron", center=CENTER, zoom=10.3, layers=list(layers)), height=740, paper_bgcolor="white",
                      font=dict(family=FONT, color=INK), margin=dict(t=95, b=55, l=10, r=10),
                      legend=dict(x=0.99, xanchor="right", y=0.98, bgcolor="rgba(255,255,255,0.9)"),
                      title=dict(text=f"<b>{title}</b><br><span style='font-size:13px;color:{INK}'>{sub}</span>", x=0.01, xanchor="left"))
    fig.add_annotation(text=note or NOTE, x=0, xref="paper", y=0, yref="paper", yanchor="top", yshift=-12, showarrow=False, font=dict(size=10.5, color="#898781"))
    return fig

# 9a. walking time to the nearest public clinic
Zc = surface(NODE_MIN["Public clinic (KKM)"])
map_fig([legend_bar(TIME_CS, 0, 60, "Minutes")] + facility_traces(["Public clinic (KKM)"]),
        "Walking time to the nearest public clinic (Klinik Kesihatan)",
        f"Along real streets and paths at {WALK_MPS} m/s, an older person's pace · lines = 15 / 30 / 45 min · shaded only where there are streets",
        [heat_layer(Zc, TIME_CS, 0, 60, [15, 30, 45])]).show()
# 9b. walking time to the nearest hospital
Zh = surface(NODE_MIN["Hospital"])
map_fig([legend_bar(TIME_CS, 0, 90, "Minutes")] + facility_traces(["Hospital"]),
        "Walking time to the nearest hospital", f"Public and private hospitals · at {WALK_MPS} m/s · lines = 30 / 60 min",
        [heat_layer(Zh, TIME_CS, 0, 90, [30, 60])]).show()
# 9b2. DRIVING time maps
for ty, vmax, lv in [("Public clinic (KKM)", 20, [5, 10, 15]), ("Hospital", 30, [10, 20])]:
    Zd = surface(DNODE_MIN[ty], drive=True)
    map_fig([legend_bar(TIME_CS, 0, vmax, "Minutes")] + facility_traces([ty]),
            f"Driving time to the nearest {'public clinic' if ty != 'Hospital' else 'hospital'}",
            f"By car on real roads, one-way streets respected · posted limits where mapped, typical speeds elsewhere, no traffic · +{ACCESS_MIN:.0f} min to park · lines = {' / '.join(map(str, lv))} min",
            [heat_layer(Zd, TIME_CS, 0, vmax, lv)]).show()

# 9c. where older residents live (smooth density)
map_fig([go.Densitymap(lat=H.lat, lon=H.lon, z=H.p65, radius=14, colorscale=HEAT_CS, opacity=0.85, hoverinfo="skip",
                       colorbar=dict(title=dict(text="Density of 65+", side="top"), thickness=10, len=0.4, x=0.01, xanchor="left", y=0.97, yanchor="top", outlinewidth=0, showticklabels=False))]
        + facility_traces(), f"Where Ipoh's {H.p65.sum():,.0f} residents aged 65+ live", "Smoothed density · with public clinics and hospitals").show()
# 9d. older residents beyond a 30-minute walk of a public clinic
H["p65_far"] = H.p65 * far
map_fig([go.Densitymap(lat=H.lat, lon=H.lon, z=H.p65_far, radius=14, colorscale=HEAT_CS, opacity=0.85, hoverinfo="skip",
                       colorbar=dict(title=dict(text="Density", side="top"), thickness=10, len=0.4, x=0.01, xanchor="left", y=0.97, yanchor="top", outlinewidth=0, showticklabels=False))]
        + facility_traces(["Public clinic (KKM)"]), f"{H.p65_far.sum():,.0f} residents aged 65+ live more than a 30-minute walk from a public clinic",
        "Smoothed density of those residents · the gaps a mobile clinic or new KK would fill").show()
# 9e. public-clinic capacity per resident
Zk = surface(NODE_CAP); kmax = float(np.nanpercentile(Zk, 97))
map_fig([legend_bar(CAP_CS, 0, kmax, "Clinics / 10k")] + facility_traces(["Public clinic (KKM)"]),
        "Public-clinic capacity per resident (E2SFCA)",
        f"Clinics within a {E2SFCA_M / 1000:.0f} km walk, shared among everyone who can reach them · each clinic counted once (no public staffing data)",
        [heat_layer(Zk, CAP_CS, 0, kmax)]).show()

# 9f. distribution: % of older residents within X minutes
xs = np.arange(0, 121)
fig = go.Figure()
for ty, col in [("Public clinic (KKM)", "#1c5cab"), ("Hospital", "#14171C")]:
  for mode, dash, marks in [("min_", "solid", (15, 30)), ("drive_", "dash", (10,))]:
    o = H.sort_values(mode + ty); cum = 100 * o.p65.cumsum() / o.p65.sum()
    yv = np.interp(xs, o[mode + ty].values, cum.values, left=0, right=100)
    nm_ = f"{ty} · {'walking' if mode == 'min_' else 'driving'}"
    fig.add_trace(go.Scatter(x=xs, y=yv, mode="lines", name=nm_, line=dict(color=col, width=2.5, dash=dash),
                             hovertemplate="%{y:.0f}% of 65+ within %{x} min<extra>" + nm_ + "</extra>"))
    for t30 in marks:
        v = float(np.interp(t30, o[mode + ty].values, cum.values, left=0, right=100))
        fig.add_trace(go.Scatter(x=[t30], y=[v], mode="markers+text", marker=dict(size=8, color=col), text=[f"{v:.0f}%"],
                                 textposition="top left", showlegend=False, hoverinfo="skip"))
fig.update_layout(height=460, paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color=INK), margin=dict(t=95, b=60, l=70, r=30),
                  legend=dict(x=0.99, xanchor="right", y=0.05, yanchor="bottom"),
                  title=dict(text=f"<b>How quickly can older residents reach care?</b><br><span style='font-size:13px'>Share of Ipoh's {H.p65.sum():,.0f} residents aged 65+ within each time · solid = walking at {WALK_MPS} m/s, dashed = driving</span>", x=0.01, xanchor="left"),
                  xaxis=dict(showgrid=False, zeroline=False, title="Travel time (minutes)", linecolor="#c3c2b7", range=[0, 120]),
                  yaxis=dict(showgrid=False, zeroline=False, ticksuffix="%", range=[0, 102], linecolor="#c3c2b7"))
fig.show()

# ---- 10. priority gaps + save --------------------------------------------------------------------------
top = H[far].sort_values("p65", ascending=False).head(15)
print("Hexagons with the most residents aged 65+ beyond a 30-min walk of a public clinic:")
display(top[["hex_id", "lat", "lon", "pop", "p65", "min_Public clinic (KKM)", "drive_Public clinic (KKM)", "min_Hospital", "drive_Hospital", "clinics_per_10k"]].round(1))
out = H.copy(); out["walk_speed_mps"] = WALK_MPS
out.round(2).to_csv("outputs/ipoh_health_access_by_hex.csv", index=False)
A.drop(columns=["x", "y", "node", "dnode"]).to_csv("outputs/ipoh_health_facilities.csv", index=False)
print("Saved outputs/ipoh_health_access_by_hex.csv and outputs/ipoh_health_facilities.csv")

def hex_surface(vals):
    """Smooth surface from hexagon-centre values (same look as the travel-time maps)."""
    v = np.asarray(vals, float); ok = np.isfinite(v)
    Z = griddata(np.c_[hx[ok], hy[ok]], v[ok], (gx, gy), method="linear").reshape(GLON.shape)
    Zn = griddata(np.c_[hx[ok], hy[ok]], v[ok], (gx, gy), method="nearest").reshape(GLON.shape)
    return np.where(MASK, np.where(np.isfinite(Z), Z, Zn), np.nan)
HX_HEALTH, HY_HEALTH = hx.copy(), hy.copy()

People: 772,527 in 841 hexagons · aged 65+: 74,207
   busy/failed: HTTP 502
OSM: 137 health features
Healthsites: malaysia-healthsites-csv-with-hxl-tags
Healthsites: 100 in the area, 15 not already in OSM by id
  + added from KKM list: Klinik Kesihatan Chemor (geocoded from 'Jalan Tanah Hitam, 31200 Chemor, Perak')
  + added from KKM list: Klinik Kesihatan UTC Ipoh (geocoded from 'UTC Perak, Jalan Onn Jaafar, 30300 Ipoh, Perak')

Facilities used (after de-duplication):


src,KKM list,OSM
type,,
Hospital,0,15
Public clinic (KKM),2,17


Public clinics used — CHECK against KKM's list for Kinta; add any missing ones to KKM_OFFICIAL:
  Klinik 1Malaysia Meru Jaya
  Klinik Desa Bemban
  Klinik Desa Tambun
  Klinik Kesihatan Buntong
  Klinik Kesihatan Chemor
  Klinik Kesihatan Greentown
  Klinik Kesihatan Gunung Rapat
  Klinik Kesihatan Ibu dan Anak Pengkalan Pegoh
  Klinik Kesihatan Ibu dan Anak Simpang Pulai
  Klinik Kesihatan Jelapang
  Klinik Kesihatan Kampung Sungai Itek
  Klinik Kesihatan Manjoi
  Klinik Kesihatan Menglembu
  Klinik Kesihatan Pasir Pinji
  Klinik Kesihatan Simee
  Klinik Kesihatan Sungai Siput (U)
  Klinik Kesihatan Tanjung Rambutan
  Klinik Kesihatan UTC Ipoh
  Klinik Komuniti Bercham
   busy/failed: HTTP 502
   busy/failed: HTTP 502
   busy/failed: HTTP 504
   busy/failed: HTTP 504
All servers busy — waiting 30s and trying again (1/3)

Walking network: 202,259 nodes, 225,558 segments, 6,936 km · main connected part 98% of nodes
Hexagon centres snapped to the network: median 40 m, 95th pct 303 m

Dri

,< 15 min,15–30 min,30–45 min,> 45 min
Public clinic (KKM),5.7,21.5,26.0,46.8
Hospital,5.1,6.0,13.3,75.6


Share of residents aged 65+ by DRIVING time (incl. 2 min access, no traffic):


,< 5 min,5–10 min,10–15 min,15–20 min,> 20 min
Public clinic (KKM),21.1,62.8,14.1,1.6,0.4
Hospital,7.1,46.9,38.4,5.8,1.9


→ 54,039 residents aged 65+ (73%) are more than a 30-minute walk from a public clinic.


Hexagons with the most residents aged 65+ beyond a 30-min walk of a public clinic:


,hex_id,lat,lon,pop,p65,min_Public clinic (KKM),drive_Public clinic (KKM),min_Hospital,drive_Hospital,clinics_per_10k
360,H-002-001,4.6,101.1,2242.6,313.3,44.6,6.8,56.3,7.4,0.1
509,H+002+001,4.6,101.1,2113.5,295.3,30.2,5.0,13.4,4.6,0.2
508,H+002+000,4.6,101.1,1886.2,263.5,35.0,5.9,28.8,5.8,0.3
539,H+003-004,4.6,101.1,2322.4,260.9,41.4,5.8,3.7,3.3,0.2
540,H+003-003,4.6,101.1,1861.6,260.1,41.4,5.8,3.7,3.3,0.2
472,H+001+000,4.6,101.1,1843.2,257.5,35.0,5.9,28.8,5.8,0.3
435,H+000+001,4.6,101.1,1812.5,253.2,34.6,8.1,37.8,9.3,0.3
473,H+001+001,4.6,101.1,1806.3,252.4,35.6,4.9,30.8,4.8,0.4
544,H+003+001,4.6,101.1,2635.8,242.5,30.2,5.0,13.4,4.6,0.2
674,H+007-005,4.6,101.1,2623.5,241.4,31.8,8.7,62.2,8.7,0.2


Saved outputs/ipoh_health_access_by_hex.csv and outputs/ipoh_health_facilities.csv


 2b · Hospital beds per 10,000 people (Kinta vs Perak vs Malaysia)

In [ ]:
# ==================================================================================================
# HOSPITAL BEDS PER 10,000 PEOPLE — Kinta (Ipoh's district) vs Perak vs Malaysia (single cell, real data only)
#   Beds      : data.gov.my "hospital_beds" (MOH Health Informatics Centre; active beds at 31 Dec; public sector)
#   People    : OpenDOSM population by district / state / Malaysia (same year as the beds where possible)
#   Private   : not in that file → optional table you fill from each hospital's own published figure (no guessing)
#   Benchmark : same indicator as World Bank SH.MED.BEDS.ZS (beds per 1,000) — × 10 for per 10,000
# ==================================================================================================
import os, re, numpy as np, pandas as pd, plotly.graph_objects as go
from IPython.display import display

BEDS_URL = "https://storage.data.gov.my/healthcare/hospital_beds.parquet"
POP = {"district": "https://storage.dosm.gov.my/population/population_district.parquet",
       "state":    "https://storage.dosm.gov.my/population/population_state.parquet",
       "malaysia": "https://storage.dosm.gov.my/population/population_malaysia.parquet"}
DISTRICT, STATE = "Kinta", "Perak"
# Private hospitals in Ipoh: fill beds from each hospital's website / annual report / MOH licence list, with the source.
# Left empty = private beds are NOT counted (and the output says so). Example row format:
#   "KPJ Ipoh Specialist Hospital": (beds, "source URL"),
PRIVATE_BEDS = {}

def rd(u):
    d = pd.read_parquet(u) if u.endswith(".parquet") else pd.read_csv(u)
    if "date" in d: d["date"] = pd.to_datetime(d["date"])
    return d
B = rd(BEDS_URL)
print("Bed years available:", sorted(B.date.dt.year.unique()), "· types:", sorted(B.type.unique()))
YEAR = int(B.date.dt.year.max())

def pop_of(level, name=None):
    d = rd(POP[level])
    for c, v in (("sex", "both"), ("ethnicity", "overall")):
        if c in d: d = d[d[c] == v]
    if name is not None:
        col = "district" if level == "district" else "state"
        d = d[d[col].str.lower() == name.lower()]
    yrs = sorted(d.date.dt.year.unique()); y = YEAR if YEAR in yrs else max(yrs)
    d = d[d.date.dt.year == y]
    tot = float(d.loc[d.age == "overall", "population"].sum())
    lo = d.age.astype(str).str.extract(r"^(\d+)")[0].astype(float)
    p65 = float(d.loc[lo >= 65, "population"].sum())
    return tot * 1000, p65 * 1000, y          # OpenDOSM population is in thousands

def beds_of(state, district):
    d = B[(B.date.dt.year == YEAR) & (B.state == state) & (B.district == district)]
    return d.groupby("type").beds.sum()
rows = []
for label, st, di, lvl, nm in [(f"{DISTRICT} district", STATE, DISTRICT, "district", DISTRICT),
                               (f"{STATE} state", STATE, "All Districts", "state", STATE),
                               ("Malaysia", "Malaysia", "All Districts", "malaysia", None)]:
    b = beds_of(st, di)
    if b.empty: print(f"⚠ no bed rows for {label} — check spelling in B.state / B.district"); continue
    pop, p65, py = pop_of(lvl, nm)
    rows.append(dict(area=label, MOH_hospitals=b.get("hospital_moh", 0), special_institutions=b.get("special_medical_institution", 0),
                     non_MOH_public=b.get("hospital_non_moh", 0), population=pop, aged_65=p65, pop_year=py))
T = pd.DataFrame(rows).set_index("area")
T["public_beds"] = T[["MOH_hospitals", "special_institutions", "non_MOH_public"]].sum(axis=1)
T["general_beds"] = T.MOH_hospitals + T.non_MOH_public                     # without psychiatric/leprosy/etc. institutions
T["beds_per_10k"] = 1e4 * T.public_beds / T.population
T["general_beds_per_10k"] = 1e4 * T.general_beds / T.population
T["beds_per_1k_aged65"] = 1e3 * T.public_beds / T.aged_65

if PRIVATE_BEDS:
    pv = sum(v[0] for v in PRIVATE_BEDS.values())
    T.loc[f"{DISTRICT} district", "private_beds"] = pv
    T["all_beds_per_10k"] = 1e4 * (T.public_beds + T.private_beds.fillna(np.nan)) / T.population
    print(f"Private beds added for {DISTRICT}: {pv:,} from {len(PRIVATE_BEDS)} hospitals (sources in PRIVATE_BEDS)")
else:
    print("Private hospital beds NOT included (PRIVATE_BEDS is empty) → figures are PUBLIC beds only.")
print(f"\nPublic hospital beds, {YEAR} (data.gov.my) · population {T.pop_year.iloc[0]} (OpenDOSM)")
display(T.drop(columns="pop_year").round(1))

# ---- chart ------------------------------------------------------------------------------------------
FONT = "Inter, system-ui, sans-serif"; INK = "#52514e"
fig = go.Figure()
for col, name, colr in [("general_beds_per_10k", "General hospitals", "#1c5cab"), ("beds_per_10k", "+ special institutions (psychiatric etc.)", "#86b6ef")]:
    fig.add_trace(go.Bar(x=T.index, y=T[col], name=name, marker=dict(color=colr, cornerradius=4),
                         text=[f"{v:.1f}" for v in T[col]], textposition="outside", hovertemplate="%{x}: %{y:.1f} beds per 10,000<extra>" + name + "</extra>"))
fig.update_layout(barmode="group", height=440, paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color=INK),
                  margin=dict(t=100, b=50, l=50, r=30), legend=dict(orientation="h", y=1.04, x=0),
                  title=dict(text=f"<b>Public hospital beds per 10,000 people, {YEAR}</b><br><span style='font-size:13px'>MOH via data.gov.my · population OpenDOSM · private hospitals {'included for Kinta' if PRIVATE_BEDS else 'not included'}</span>", x=0.01, xanchor="left"),
                  xaxis=dict(showgrid=False, linecolor="#c3c2b7"), yaxis=dict(showgrid=False, visible=False))
fig.show()
print("Note: Kinta's main hospital (Hospital Raja Permaisuri Bainun) is the referral centre for the wider Ipoh cluster,"
      " so Kinta's ratio overstates beds available to Kinta residents alone. Compare Kinta with Perak and Malaysia, not with India.")

Bed years available: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)] · types: ['all', 'hospital_moh', 'hospital_non_moh', 'special_medical_institution']
⚠ no bed rows for Kinta district — check spelling in B.state / B.district
⚠ no bed rows for Malaysia — check spelling in B.state / B.district
Private hospital beds NOT included (PRIVATE_BEDS is empty) → figures are PUBLIC beds only.

Public hospital beds, 2024 (data.gov.my) · population 2024 (OpenDOSM)


,MOH_hospitals,special_institutions,non_MOH_public,population,aged_65,public_beds,general_beds,beds_per_10k,general_beds_per_10k,beds_per_1k_aged65
area,,,,,,,,,,
Perak state,4071,1800,150,2569400.0,254600.0,6021,4221,23.4,16.4,23.6


Note: Kinta's main hospital (Hospital Raja Permaisuri Bainun) is the referral centre for the wider Ipoh cluster, so Kinta's ratio overstates beds available to Kinta residents alone. Compare Kinta with Perak and Malaysia, not with India.


# PART 3 · MONEY — who is better off, who is worse off

**What this notebook does, in plain words**
1. **Official income by constituency.** DOSM publishes the household income of every state constituency (DUN) from its Household Income Survey (OpenDOSM `hh_income_dun`). About ten DUNs cover Ipoh, so this already splits the city into ~10 real income areas.
2. **Where each DUN is.** DUN boundary maps come from the open MECo electoral-boundaries dataset (Thevesh/paper-meco-maps on GitHub; Harvard Dataverse doi:10.7910/DVN/DVFK54).
3. **House prices by taman.** NAPIC (JPPH) publishes every stamped house resale with its scheme name, type, size and price. Grouped by taman, these show which neighbourhoods are richer or poorer *inside* each DUN. The tamans are placed on the map by matching their names to OpenStreetMap.
4. **Spreading DUN income inside each DUN** using those house prices, then scaling so every DUN still matches DOSM's official figure. How strongly income follows house prices is **measured from the data** (across DUNs), not assumed.
5. **Cross-check** against Meta's Relative Wealth Index (2.4 km tiles).
6. **The link to ageing:** how many people aged 65+ live in each income band (uses `outputs/ipoh_pop65_by_hex.csv` from the population notebook).

**What is official and what is an estimate**
* DUN income figures: official (DOSM survey).
* Income for each 400 m hexagon: **an estimate**. Every hexagon is labelled with where its number came from (`ses_source`).
* Nothing here is simulated. If a file can't be found, the notebook stops and says what's missing. Unmatched tamans are listed, not guessed.

Needs: `pandas numpy pyarrow plotly matplotlib requests scipy openpyxl`.

In [ ]:
import os, re, io, json, math, glob, time, difflib
import numpy as np, pandas as pd
import plotly.graph_objects as go
from matplotlib.path import Path
from scipy import stats
from IPython.display import display

# ---- inputs (edit if needed) ---------------------------------------------------------------------
INCOME_DUN      = "https://storage.dosm.gov.my/hies/hh_income_dun.parquet"        # official DUN household income
INCOME_DISTRICT = "https://storage.dosm.gov.my/hies/hh_income_district.parquet"   # Kinta figure, for reference
HEX_CSV         = "outputs/ipoh_pop65_by_hex.csv"   # from ipoh_population_65plus.ipynb (hex_id, lat, lon, pop, p65, ...)
OSM_CACHE       = "data/osm_ipoh.json"              # optional: Ipoh boundary outline for the maps
DUN_GEOJSON     = "data/dun_boundaries.geojson"     # downloaded automatically from GitHub if missing (Section 2)
DUN_REPO        = "Thevesh/paper-meco-maps"
DUN_PATH_IN_REPO = None                             # set this if the automatic file pick (printed in Section 2) is wrong
NAPIC_DIR       = "data/napic"                      # put your NAPIC Open Transaction Data exports here (.csv / .xlsx)
NAPIC_COLMAP    = {}                                # override column detection, e.g. {"price": "Transaction Price"}
NAPIC_MANUAL_LOC = "data/napic_manual_locations.csv"  # optional: scheme,lat,lon for tamans OSM can't find
OSM_NAMES_CACHE = "data/osm_ipoh_names.json"
RWI_FILE        = "data/mys_relative_wealth_index.csv"   # downloaded automatically from HDX if missing
GEOCODE_UNMATCHED = False                           # True → ask OpenStreetMap's Nominatim for unmatched tamans (1 per second)
BBOX = (4.45, 100.98, 4.82, 101.32)                 # south, west, north, east
YEAR_FROM = 2021                                    # use NAPIC sales from this year on
VALUE = "price"                                     # "price" (size + location) or "psf" (location only)
MIN_SCHEME_SALES = 5                                # a taman needs at least this many sales to be used
KERNEL_M = 600                                      # how far a taman's prices "reach" (Gaussian sigma, metres)
MIN_EFF_SALES = 10                                  # a hexagon needs this much nearby sales evidence, else it keeps its DUN figure
BETA_OVERRIDE = None                                # leave None: the income–house-price link is measured in Section 5
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True); os.makedirs(NAPIC_DIR, exist_ok=True)

LAT0, LON0 = 4.60, 101.09
KX, KY = 111_320 * math.cos(math.radians(LAT0)), 110_574
HEX_R = 400.0
def to_xy(lat, lon): return (np.asarray(lon, float) - LON0) * KX, (np.asarray(lat, float) - LAT0) * KY
def to_ll(x, y): return LAT0 + np.asarray(y) / KY, LON0 + np.asarray(x) / KX
def read_table(src):
    s = str(src).lower()
    return pd.read_parquet(src) if s.endswith(".parquet") else pd.read_csv(src)
def norm_dun(s):
    """'N.25 Canning' → 'canning'"""
    s = re.sub(r"^\s*n\.?\s*\d+[a-z]?\s*[-:.]?\s*", "", str(s).lower())
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

## 1 · Official household income by DUN (DOSM)

In [ ]:
inc = read_table(INCOME_DUN); inc["date"] = pd.to_datetime(inc["date"])
per = inc[inc["state"].str.lower() == "perak"].copy()
assert len(per), "No Perak rows in the DUN income file."
per["year"] = per.date.dt.year; per["key"] = per.dun.map(norm_dun)
YEARS = sorted(per.year.unique()); LATEST = int(max(YEARS))
print("Years in the file:", YEARS, "→ using", LATEST)
INC = per[per.year == LATEST].set_index("key")[["parlimen", "dun", "income_median", "income_mean"]]

dist = read_table(INCOME_DISTRICT); dist["date"] = pd.to_datetime(dist["date"])
kin = dist[dist.district.str.lower() == "kinta"].sort_values("date")
KINTA_MEDIAN = float(kin.income_median.iloc[-1]); KINTA_YEAR = int(kin.date.dt.year.iloc[-1])
print(f"Kinta district median household income {KINTA_YEAR}: RM{KINTA_MEDIAN:,.0f}")

IPOH_PARL = ("P.062", "P.063", "P.064", "P.065", "P.066")   # Sungai Siput, Tambun, Ipoh Timor, Ipoh Barat, Batu Gajah
show = per[per.parlimen.str[:5].isin(IPOH_PARL)].pivot_table(index=["parlimen", "dun"], columns="year", values="income_median")
display(show.style.format("RM{:,.0f}"))

Years in the file: [np.int32(2019), np.int32(2022), np.int32(2024)] → using 2024
Kinta district median household income 2024: RM5,136


## 2 · DUN boundaries (MECo open electoral boundaries)
The notebook lists the GeoJSON files in the GitHub repo and picks the **latest Peninsular/Perak DUN** file (Perak's current DUNs date from the 2018 delimitation). Check the printed choice; if it is wrong, set `DUN_PATH_IN_REPO`. If GitHub is unreachable, download from Harvard Dataverse (doi:10.7910/DVN/DVFK54) and save it as `DUN_GEOJSON`.

In [ ]:
import requests
def fetch_dun_geojson():
    api = f"https://api.github.com/repos/{DUN_REPO}"
    branch = requests.get(api, timeout=30).json().get("default_branch", "main")
    tree = requests.get(f"{api}/git/trees/{branch}?recursive=1", timeout=60).json()
    paths = [t["path"] for t in tree.get("tree", []) if t["type"] == "blob" and t["path"].lower().endswith(".geojson") and "dun" in t["path"].lower()]
    assert paths, f"No DUN GeoJSON found in {DUN_REPO}. Download it from doi:10.7910/DVN/DVFK54 and save as {DUN_GEOJSON}."
    print(f"{len(paths)} DUN GeoJSON files in the repo, e.g.:", *paths[:12], sep="\n  ")
    def score(p):
        pl = p.lower(); yrs = [int(y) for y in re.findall(r"(19\d\d|20\d\d)", pl) if int(y) <= 2026]
        region = 2 if "perak" in pl else (1 if any(k in pl for k in ("peninsular", "semenanjung", "malaya")) else 0)
        return (region, max(yrs) if yrs else 0, "delimit" in pl)
    pick = DUN_PATH_IN_REPO or max(paths, key=score)
    print("→ using:", pick)
    raw = requests.get(f"https://raw.githubusercontent.com/{DUN_REPO}/{branch}/{pick}", timeout=300)
    if raw.text.startswith("version https://git-lfs"):
        raw = requests.get(f"https://media.githubusercontent.com/media/{DUN_REPO}/{branch}/{pick}", timeout=300)
    raw.raise_for_status(); open(DUN_GEOJSON, "w", encoding="utf-8").write(raw.text)

if not os.path.exists(DUN_GEOJSON):
    fetch_dun_geojson()
GJ = json.load(open(DUN_GEOJSON, encoding="utf-8"))

# which property holds the DUN name? → the one whose values best match DOSM's Perak DUN names
props = pd.DataFrame([f.get("properties", {}) for f in GJ["features"]])
best = max(props.columns, key=lambda c: props[c].astype(str).map(norm_dun).isin(INC.index).sum())
print("DUN name field:", best, "· sample:", props[best].head(3).tolist())

def polys(geom):
    if geom["type"] == "Polygon": return [geom["coordinates"]]
    if geom["type"] == "MultiPolygon": return geom["coordinates"]
    return []
# keep only DUNs near Ipoh (names repeat across states)
s, w, n, e = BBOX
DUNS = {}
for f, k in zip(GJ["features"], props[best].astype(str).map(norm_dun)):
    if k not in INC.index: continue
    P = polys(f["geometry"]); pts = np.vstack([np.asarray(r[0])[:, :2] for r in P])
    if pts[:, 0].max() < w - 0.1 or pts[:, 0].min() > e + 0.1 or pts[:, 1].max() < s - 0.1 or pts[:, 1].min() > n + 0.1: continue
    DUNS.setdefault(k, []).extend(P)
print(f"{len(DUNS)} Perak DUNs near Ipoh:", ", ".join(INC.loc[list(DUNS), "dun"]))

19 DUN GeoJSON files in the repo, e.g.:
  data/geojson/delimitations/peninsular_1958_dun.geojson
  data/geojson/delimitations/peninsular_1974_dun.geojson
  data/geojson/delimitations/peninsular_1984_dun.geojson
  data/geojson/delimitations/peninsular_1994_dun.geojson
  data/geojson/delimitations/peninsular_2003_dun.geojson
  data/geojson/delimitations/peninsular_2018_dun.geojson
  data/geojson/delimitations/sabah_1966_dun.geojson
  data/geojson/delimitations/sabah_1974_dun.geojson
  data/geojson/delimitations/sabah_1984_dun.geojson
  data/geojson/delimitations/sabah_1994_dun.geojson
  data/geojson/delimitations/sabah_2003_dun.geojson
  data/geojson/delimitations/sabah_2019_dun.geojson
→ using: data/geojson/delimitations/peninsular_2018_dun.geojson
DUN name field: dun · sample: ['N.01 Titi Tinggi', 'N.02 Beseri', 'N.03 Chuping']
26 Perak DUNs near Ipoh: N.19 Chenderoh, N.20 Lubok Merbau, N.21 Lintang, N.22 Jalong, N.23 Manjoi, N.24 Hulu Kinta, N.25 Canning, N.26 Tebing Tinggi, N.27 Pasi

### Put each hexagon into its DUN

In [ ]:
H = pd.read_csv(HEX_CSV)
need = {"hex_id", "lat", "lon", "pop", "p65"}; assert need <= set(H.columns), f"{HEX_CSV} needs columns {need}"
pts = np.c_[H.lon, H.lat]; H["dun_key"] = None
for k, P in DUNS.items():
    inside = np.zeros(len(H), bool)
    for poly in P:
        m = Path(np.asarray(poly[0])[:, :2]).contains_points(pts)
        for hole in poly[1:]: m &= ~Path(np.asarray(hole)[:, :2]).contains_points(pts)
        inside |= m
    H.loc[inside & H.dun_key.isna(), "dun_key"] = k
miss = H.dun_key.isna()
print(f"{(~miss).sum():,} of {len(H):,} hexagons placed in a DUN ({100 * H.loc[~miss, 'pop'].sum() / H['pop'].sum():.1f}% of people).")
if miss.any():   # edge hexagons whose centre falls just outside every polygon → nearest DUN hexagon (reported)
    from scipy.spatial import cKDTree
    ok = H[~miss]; t = cKDTree(np.c_[to_xy(ok.lat, ok.lon)])
    dd, ii = t.query(np.c_[to_xy(H.loc[miss, "lat"], H.loc[miss, "lon"])])
    H.loc[miss, "dun_key"] = ok.dun_key.values[ii]
    print(f"  {miss.sum()} edge hexagons (max {dd.max():.0f} m outside) given their nearest DUN.")
H["dun"] = INC.loc[H.dun_key, "dun"].values
H["dun_median"] = INC.loc[H.dun_key, "income_median"].values
D = (H.groupby("dun_key").agg(pop_in_ipoh=("pop", "sum"), p65_in_ipoh=("p65", "sum"), lat=("lat", "mean"), lon=("lon", "mean"))
       .join(INC).sort_values("income_median"))
display(D[["parlimen", "dun", "income_median", "income_mean", "pop_in_ipoh", "p65_in_ipoh"]].round(0))

841 of 841 hexagons placed in a DUN (100.0% of people).


,parlimen,dun,income_median,income_mean,pop_in_ipoh,p65_in_ipoh
dun_key,,,,,,
pasir pinji,P.064 Ipoh Timor,N.27 Pasir Pinji,3165.0,4704.0,41478.0,4711.0
menglembu,P.066 Batu Gajah,N.32 Menglembu,3943.0,5824.0,49244.0,4560.0
bercham,P.065 Ipoh Barat,N.28 Bercham,4417.0,5103.0,51056.0,4697.0
tebing tinggi,P.064 Ipoh Timor,N.26 Tebing Tinggi,4434.0,5432.0,54153.0,5400.0
buntong,P.065 Ipoh Barat,N.30 Buntong,4619.0,5184.0,30683.0,2985.0
jelapang,P.066 Batu Gajah,N.31 Jelapang,4623.0,5646.0,28506.0,2623.0
sungai rapat,P.071 Gopeng,N.44 Sungai Rapat,4988.0,6239.0,53882.0,4957.0
teja,P.071 Gopeng,N.46 Teja,5145.0,5473.0,1819.0,167.0
manjoi,P.063 Tambun,N.23 Manjoi,5528.0,7554.0,160640.0,14779.0


### Maps: official DUN income

In [ ]:
FONT = "Inter, system-ui, sans-serif"
BLUES = [[0, "#eef4fc"], [0.3, "#cde2fb"], [0.55, "#86b6ef"], [0.8, "#1c5cab"], [1, "#0d366b"]]
CENTER = dict(lat=float(H.lat.mean()), lon=float(H.lon.mean()))
RINGS = []
if os.path.exists(OSM_CACHE):
    def stitch(segments):
        segs = [list(s) for s in segments if len(s) >= 2]; rings = []
        while segs:
            ring = segs.pop(0); changed = True
            while ring[0] != ring[-1] and changed:
                changed = False
                for i, s in enumerate(segs):
                    if s[0] == ring[-1]: ring += s[1:]
                    elif s[-1] == ring[-1]: ring += s[::-1][1:]
                    elif s[-1] == ring[0]: ring = s[:-1] + ring
                    elif s[0] == ring[0]: ring = s[::-1][:-1] + ring
                    else: continue
                    segs.pop(i); changed = True; break
            if len(ring) >= 4 and ring[0] == ring[-1]: rings.append(np.array(ring, float))
        return rings
    for el in json.load(open(OSM_CACHE, encoding="utf-8")).get("elements", []):
        t = el.get("tags", {})
        if el.get("type") == "relation" and t.get("boundary") == "administrative" and "ipoh" in str(t.get("name", "")).lower():
            RINGS += stitch([[(p["lat"], p["lon"]) for p in m["geometry"]] for m in el.get("members", []) if m.get("role") in ("outer", "") and m.get("geometry")])

def hex_ring(lat, lon, R=HEX_R):
    cx, cy = to_xy(lat, lon); a = np.deg2rad(np.arange(30, 390, 60))
    la, lo = to_ll(cx + R * np.cos(a), cy + R * np.sin(a))
    ring = [[round(float(b), 6), round(float(c), 6)] for c, b in zip(la, lo)]; return ring + [ring[0]]
HGEO = {"type": "FeatureCollection", "features": [{"type": "Feature", "id": h, "properties": {"id": h},
        "geometry": {"type": "Polygon", "coordinates": [hex_ring(a, b)]}} for h, a, b in zip(H.hex_id, H.lat, H.lon)]}
DGEO = {"type": "FeatureCollection", "features": [{"type": "Feature", "id": k, "properties": {"id": k},
        "geometry": {"type": "MultiPolygon", "coordinates": P}} for k, P in DUNS.items()]}

def base_layout(fig, title, sub, note, height=720):
    for ring in RINGS:
        fig.add_trace(go.Scattermap(lat=ring[:, 0], lon=ring[:, 1], mode="lines", line=dict(width=2, color="#14171C"), hoverinfo="skip", showlegend=False))
    fig.update_layout(map=dict(style="carto-positron", center=CENTER, zoom=10.2), height=height, paper_bgcolor="white",
                      font=dict(family=FONT, color="#52514e"), margin=dict(t=95, b=55, l=10, r=10),
                      title=dict(text=f"<b>{title}</b><br><span style='font-size:13px;color:#52514e'>{sub}</span>", x=0.01, xanchor="left"))
    fig.add_annotation(text=note, x=0, xref="paper", y=0, yref="paper", yanchor="top", yshift=-12, showarrow=False, font=dict(size=10.5, color="#898781"), align="left")
    return fig
def cbar(label): return dict(title=dict(text=label, side="top"), thickness=10, len=0.45, x=0.01, xanchor="left", y=0.97, yanchor="top", outlinewidth=0, bgcolor="rgba(255,255,255,0.85)")
SRC_DUN = f"Household income: DOSM HIES via OpenDOSM hh_income_dun ({LATEST}) · DUN boundaries: MECo (Thevesh/paper-meco-maps) · © OpenStreetMap contributors"

zmin, zmax = float(D.income_median.min()), float(D.income_median.max())
fig = go.Figure(go.Choroplethmap(geojson=DGEO, locations=D.index, z=D.income_median, colorscale=BLUES, zmin=zmin, zmax=zmax,
                marker=dict(opacity=0.75, line=dict(width=1.2, color="white")), colorbar=cbar("RM / month"),
                customdata=np.stack([D.dun, D.parlimen, D.income_mean, D.pop_in_ipoh], -1),
                hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>Median: RM%{z:,.0f}<br>Mean: RM%{customdata[2]:,.0f}<br>People in Ipoh: %{customdata[3]:,.0f}<extra></extra>"))
fig.add_trace(go.Scattermap(lat=D.lat, lon=D.lon, mode="text", text=[f"<b>{d.split(' ', 1)[-1]}</b><br>RM{v:,.0f}" for d, v in zip(D.dun, D.income_median)],
                            textfont=dict(size=11, color="#14171C"), hoverinfo="skip", showlegend=False))
base_layout(fig, f"Median household income by state constituency, {LATEST}", f"Official DOSM survey figures · Kinta district median RM{KINTA_MEDIAN:,.0f} ({KINTA_YEAR})", SRC_DUN).show()

In [ ]:
bar = D.sort_values("income_median")
fig = go.Figure()
first = per[per.year == min(YEARS)].set_index("key").income_median.reindex(bar.index)
fig.add_trace(go.Scatter(x=first, y=bar.dun, mode="markers", name=str(min(YEARS)), marker=dict(size=10, color="#c3c2b7"),
                         hovertemplate="%{y}: RM%{x:,.0f}<extra>" + str(min(YEARS)) + "</extra>"))
fig.add_trace(go.Scatter(x=bar.income_median, y=bar.dun, mode="markers+text", name=str(LATEST), marker=dict(size=11, color="#1c5cab"),
                         text=[f"RM{v:,.0f}" for v in bar.income_median], textposition="middle right", hovertemplate="%{y}: RM%{x:,.0f}<extra>" + str(LATEST) + "</extra>"))
for y, a, b in zip(bar.dun, first, bar.income_median):
    fig.add_shape(type="line", x0=a, x1=b, y0=y, y1=y, line=dict(color="#c3c2b7", width=2), layer="below")
fig.add_shape(type="line", x0=KINTA_MEDIAN, x1=KINTA_MEDIAN, y0=-0.5, y1=len(bar) - 0.5, line=dict(color="#14171C", width=1, dash="dot"))
fig.add_annotation(text=f"Kinta {KINTA_YEAR}", x=KINTA_MEDIAN, y=len(bar) - 0.5, yanchor="bottom", showarrow=False, font=dict(size=11))
fig.update_layout(height=120 + 34 * len(bar), paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color="#52514e"),
                  margin=dict(t=90, b=50, l=170, r=60), legend=dict(orientation="h", y=1.02, x=1, xanchor="right"),
                  title=dict(text=f"<b>Median household income, Ipoh DUNs: {min(YEARS)} → {LATEST}</b><br><span style='font-size:13px'>DOSM, OpenDOSM hh_income_dun · nominal RM per month</span>", x=0.01, xanchor="left"),
                  xaxis=dict(showgrid=False, zeroline=False, tickprefix="RM", linecolor="#c3c2b7"), yaxis=dict(showgrid=False))
fig.show()

## 3 · NAPIC house sales by taman
**How to get the files:** NAPIC portal (https://napic2.jpph.gov.my) → *Open Transaction Data* → filter **Perak → Kinta → Residential** → export / download, and drop every file into `data/napic/`. Several files (e.g. one per year) are fine.
Column names are detected automatically and printed; fix any wrong one with `NAPIC_COLMAP`.

In [ ]:
files = sorted(glob.glob(os.path.join(NAPIC_DIR, "*.csv")) + glob.glob(os.path.join(NAPIC_DIR, "*.xls*")))
NAPIC_OK = bool(files)
if not NAPIC_OK: print(f"No NAPIC files in {NAPIC_DIR}/. Download Perak → Kinta residential transactions from NAPIC's Open Transaction Data "
               "(https://napic2.jpph.gov.my) and put them there. → Money uses OFFICIAL DUN income only (no within-DUN estimate).")
if NAPIC_OK:
    def read_any(p):
        raw = pd.read_csv(p, header=None, dtype=str, encoding_errors="replace") if p.lower().endswith(".csv") else pd.read_excel(p, header=None, dtype=str)
        hdr = next((i for i in range(min(30, len(raw))) if raw.iloc[i].astype(str).str.lower().str.contains("price|harga").any()), 0)
        df = raw.iloc[hdr + 1:].copy(); cols, seen = [], {}
        for c in (str(c).strip() for c in raw.iloc[hdr]):        # NAPIC repeats "Unit" → make names unique
            seen[c] = seen.get(c, -1) + 1; cols.append(c if seen[c] == 0 else f"{c}.{seen[c]}")
        df.columns = cols
        return df.dropna(how="all")
    N = pd.concat([read_any(p) for p in files], ignore_index=True)
    print(f"{len(N):,} rows from {len(files)} file(s). Columns: {list(N.columns)}")

    def find(*keys, exclude=()):
        for c in N.columns:
            cl = c.lower()
            if any(k in cl for k in keys) and not any(x in cl for x in exclude): return c
        return None
    COL = {"price": find("transaction price", "harga", "price", exclude=("psf", "per ", "unit")),
           "scheme": find("scheme", "skim", "taman", "project", "projek"),
           "mukim": find("mukim"), "district": find("district", "daerah"),
           "type": find("property type", "jenis harta", "type", "jenis"),
           "area": find("main floor", "floor area", "luas lantai", "built", "keluasan lantai"),
           "land": find("land area", "luas tanah", "keluasan tanah", "parcel"),
           "year": find("year", "tahun"), "month": find("month", "bulan"), "date": find("date", "tarikh")}
    COL.update(NAPIC_COLMAP)
    print("Detected columns:", json.dumps(COL, indent=1))
    assert COL["price"] and COL["scheme"], "Couldn't find the price or scheme-name column — set NAPIC_COLMAP."

    num = lambda s: pd.to_numeric(s.astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
    N["price"] = num(N[COL["price"]])
    if COL["area"]:
        N["area"] = num(N[COL["area"]])
        ui = list(N.columns).index(COL["area"])
        unit_col = N.columns[ui + 1] if ui + 1 < len(N.columns) and "unit" in N.columns[ui + 1].lower() else None
        if unit_col:
            u = N[unit_col].astype(str).str.lower(); sqm = u.str.contains("m") & ~u.str.contains("ft|kaki")
            print("Floor-area units seen:", u.value_counts().head(5).to_dict())
        else:
            sqm = pd.Series(N["area"].median() < 400, index=N.index)
            print(f"No unit column next to floor area; median area {N['area'].median():.0f} → treated as {'m²' if sqm.iloc[0] else 'sq ft'} (CHECK THIS)")
        N["sqft"] = np.where(sqm, N["area"] * 10.7639, N["area"]); N["psf"] = N.price / N.sqft
    if COL["date"]: N["year"] = pd.to_datetime(N[COL["date"]], errors="coerce", dayfirst=True).dt.year
    elif COL["year"]: N["year"] = num(N[COL["year"]])
    if COL["district"]: N = N[N[COL["district"]].astype(str).str.lower().str.contains("kinta")]
    if "year" in N: N = N[N.year >= YEAR_FROM]
    if COL["type"]:
        bad = N[COL["type"]].astype(str).str.lower().str.contains("shop|office|industr|kedai|pejabat|agri|pertanian|vacant|kosong|factory|kilang")
        N = N[~bad]
    N = N[N.price > 10_000]
    print(f"{len(N):,} residential sales in Kinta since {YEAR_FROM} after cleaning.")

    S = (N.groupby([N[COL["scheme"]].astype(str).str.strip()] + ([N[COL["mukim"]].astype(str).str.strip()] if COL["mukim"] else []))
           .agg(n=("price", "size"), price=("price", "median"), **({"psf": ("psf", "median")} if "psf" in N else {}),
                type=(COL["type"] if COL["type"] else "price", lambda s: s.mode().iloc[0] if len(s.mode()) else ""))
           .reset_index())
    S.columns = ["scheme"] + (["mukim"] if COL["mukim"] else []) + list(S.columns[1 + bool(COL["mukim"]):])
    S = S[S.n >= MIN_SCHEME_SALES].reset_index(drop=True)
    print(f"{len(S):,} tamans with ≥{MIN_SCHEME_SALES} sales ({S.n.sum():,} sales).")
    display(S.sort_values("n", ascending=False).head(15))

No NAPIC files in data/napic/. Download Perak → Kinta residential transactions from NAPIC's Open Transaction Data (https://napic2.jpph.gov.my) and put them there. → Money uses OFFICIAL DUN income only (no within-DUN estimate).


### Put tamans on the map (OpenStreetMap names)
Names are cleaned (`Tmn`→`taman`, `Bdr`→`bandar`, brackets and "Fasa" dropped) and matched exactly first, then by close spelling (≥ 90% similar). A name that appears in two places more than 1.5 km apart is **not** used (ambiguous). Every unmatched taman is saved to `outputs/napic_unmatched_schemes.csv`; add coordinates in `data/napic_manual_locations.csv` (scheme,lat,lon) if you want them in.

In [ ]:
if NAPIC_OK:
    if os.path.exists(OSM_NAMES_CACHE):
        OSMN = json.load(open(OSM_NAMES_CACHE, encoding="utf-8"))
    else:
        bb = f"{BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]}"
        q = (f'[out:json][timeout:240];(way["landuse"="residential"]["name"]({bb});relation["landuse"="residential"]["name"]({bb});'
             f'node["place"~"^(suburb|quarter|neighbourhood|village|hamlet)$"]["name"]({bb});way["place"]["name"]({bb});'
             f'way["building"~"^(apartments|residential|condominium)$"]["name"]({bb});relation["building"~"^(apartments|residential)$"]["name"]({bb}););out center tags;')
        OSMN = requests.post("https://overpass-api.de/api/interpreter", data={"data": q}, timeout=300, headers={"User-Agent": "ipoh-ses/1.0"}).json()
        json.dump(OSMN, open(OSM_NAMES_CACHE, "w", encoding="utf-8"))

    ABBR = {"tmn": "taman", "tman": "taman", "bdr": "bandar", "bndr": "bandar", "kg": "kampung", "kpg": "kampung", "kampong": "kampung",
            "spg": "simpang", "jln": "jalan", "sg": "sungai", "sungei": "sungai", "bkt": "bukit", "sri": "seri", "psn": "persiaran",
            "pgkln": "pengkalan", "peng": "pengkalan", "tg": "tanjung", "tanjong": "tanjung", "bt": "batu", "pr": "perumahan", "rmh": "rumah"}
    def variants(s):
        s = str(s).lower(); out = []
        for v in (s, s.split(",")[0], s.split("@")[0] if len(s.split("@")[0].split()) >= 2 else s):
            v = re.sub(r"\(.*?\)|\bp/?l\b.*|\bfasa\b.*|\bphase\b.*|\bseksyen\b.*", " ", v)
            v = re.sub(r"[^a-z0-9 ]+", " ", v)
            v = " ".join(ABBR.get(t, t) for t in v.split())
            v = re.sub(r"\b(i|ii|iii|iv|v)$", "", v).strip()
            for x in (v, re.sub(r"^taman ", "", v)):
                if len(x) >= 4 and x not in out: out.append(x)
        return out
    IDX = {}
    for el in OSMN.get("elements", []):
        t = el.get("tags", {}); la = el.get("lat", el.get("center", {}).get("lat")); lo = el.get("lon", el.get("center", {}).get("lon"))
        if la is None: continue
        for tag in ("name", "alt_name", "old_name", "name:ms"):
            if tag in t:
                for v in variants(t[tag]): IDX.setdefault(v, []).append((la, lo))
    print(f"{len(IDX):,} OSM place/estate name variants around Ipoh.")
    KEYS = list(IDX)

    def locate(name):
        for v in variants(name):
            hit, how = IDX.get(v), "exact"
            if not hit:
                m = difflib.get_close_matches(v, KEYS, n=1, cutoff=0.9)
                if m: hit, how = IDX[m[0]], f"close ({m[0]})"
            if hit:
                a = np.array(hit); x, y = to_xy(a[:, 0], a[:, 1])
                if np.hypot(x - x.mean(), y - y.mean()).max() > 1500: return np.nan, np.nan, "ambiguous"
                return a[:, 0].mean(), a[:, 1].mean(), how
        return np.nan, np.nan, "unmatched"
    S[["lat", "lon", "match"]] = pd.DataFrame([locate(s) for s in S.scheme], index=S.index)

    if os.path.exists(NAPIC_MANUAL_LOC):
        man = pd.read_csv(NAPIC_MANUAL_LOC).set_index("scheme")
        m = S.scheme.isin(man.index)
        S.loc[m, ["lat", "lon"]] = man.loc[S.scheme[m], ["lat", "lon"]].values; S.loc[m, "match"] = "manual"
    if GEOCODE_UNMATCHED:
        cache_f = "data/nominatim_cache.json"; cache = json.load(open(cache_f)) if os.path.exists(cache_f) else {}
        for i in S.index[S.lat.isna()]:
            nm = S.at[i, "scheme"]
            if nm not in cache:
                r = requests.get("https://nominatim.openstreetmap.org/search", params={"q": f"{nm}, Ipoh, Perak", "format": "json", "limit": 1,
                                 "viewbox": f"{BBOX[1]},{BBOX[2]},{BBOX[3]},{BBOX[0]}", "bounded": 1}, headers={"User-Agent": "ipoh-ses/1.0"}, timeout=30).json()
                cache[nm] = r[0] if r else None; time.sleep(1.1)
            if cache[nm]: S.loc[i, ["lat", "lon", "match"]] = [float(cache[nm]["lat"]), float(cache[nm]["lon"]), "nominatim"]
        json.dump(cache, open(cache_f, "w"))

    ok = S.lat.notna()
    print(f"Placed {ok.sum()} of {len(S)} tamans ({100 * S.n[ok].sum() / S.n.sum():.0f}% of sales). How:", S.match.str.split(" ").str[0].value_counts().to_dict())
    S[~ok].sort_values("n", ascending=False).to_csv("outputs/napic_unmatched_schemes.csv", index=False)
    display(S[ok & S.match.str.startswith("close")][["scheme", "match"]].head(20))   # check the fuzzy matches by eye

In [ ]:
if NAPIC_OK:
    SP = S[ok].copy(); SP["v"] = SP[VALUE]
    fig = go.Figure(go.Scattermap(lat=SP.lat, lon=SP.lon, mode="markers",
                    marker=dict(size=np.clip(np.sqrt(SP.n) * 2.2, 6, 30), color=SP.v, colorscale=BLUES, cmin=SP.v.quantile(0.05), cmax=SP.v.quantile(0.95),
                                opacity=0.9, colorbar=cbar("RM" if VALUE == "price" else "RM / sq ft")),
                    customdata=np.stack([SP.scheme, SP.type, SP.n, SP.price], -1),
                    hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>Median price RM%{customdata[3]:,.0f}<br>%{customdata[2]} sales<extra></extra>"))
    fig.add_trace(go.Choroplethmap(geojson=DGEO, locations=D.index, z=np.zeros(len(D)), colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]], showscale=False,
                                   marker=dict(line=dict(width=1, color="#898781")), hoverinfo="skip"))
    base_layout(fig, "What homes sell for, taman by taman", f"Median resale {'price' if VALUE == 'price' else 'price per sq ft'} since {YEAR_FROM} · dot size = number of sales · grey lines = DUNs",
                "House sales: NAPIC (JPPH) Open Transaction Data · taman locations: OpenStreetMap names · © OpenStreetMap contributors").show()

## 4 · House-value level around every hexagon
Each hexagon takes a distance-weighted average of nearby tamans' (log) median value, weighted by number of sales. Hexagons without enough nearby sales evidence keep only their DUN figure.

In [ ]:
if NAPIC_OK:
    hx, hy = to_xy(H.lat, H.lon); sx, sy = to_xy(SP.lat, SP.lon)
    d2 = (hx[:, None] - sx[None]) ** 2 + (hy[:, None] - sy[None]) ** 2
    W = SP.n.values[None] * np.exp(-0.5 * d2 / KERNEL_M ** 2) * (d2 < (3 * KERNEL_M) ** 2)
    H["eff_sales"] = W.sum(1)
    H["logV"] = np.where(H.eff_sales >= MIN_EFF_SALES, (W * np.log(SP.v.values)[None]).sum(1) / np.maximum(W.sum(1), 1e-9), np.nan)
    print(f"{H.logV.notna().sum():,} hexagons ({100 * H.loc[H.logV.notna(), 'pop'].sum() / H['pop'].sum():.0f}% of people) have enough nearby sales.")

else:
    H["eff_sales"] = 0.0; H["logV"] = np.nan

## 5 · How strongly does income follow house prices? (measured, not assumed)
For each DUN: the population-weighted average house-value level of its hexagons vs its official median income. The slope (**β**) on a log–log scale says "if house values are 10% higher, household income is about β×10% higher". It is then used *inside* each DUN.
If the slope isn't clearly positive, the notebook says so and keeps the DUN figures only.

In [ ]:
if NAPIC_OK:
    g = H.dropna(subset=["logV"]).groupby("dun_key").apply(lambda t: pd.Series({"G": np.average(t.logV, weights=t["pop"] + 1e-9), "cover": t["pop"].sum()}))
    g = g.join(D[["dun", "income_median", "pop_in_ipoh"]]); g["cover"] /= g.pop_in_ipoh
    fitd = g[g.cover >= 0.3]
    if BETA_OVERRIDE is not None:
        BETA, BETA_NOTE = float(BETA_OVERRIDE), f"β = {BETA_OVERRIDE} SET BY HAND"
    elif len(fitd) >= 5:
        r = stats.linregress(fitd.G, np.log(fitd.income_median))
        lo, hi = r.slope + np.array([-1, 1]) * stats.t.ppf(0.975, len(fitd) - 2) * r.stderr
        print(f"β = {r.slope:.2f} (95% CI {lo:.2f} to {hi:.2f}), R² = {r.rvalue ** 2:.2f}, from {len(fitd)} DUNs")
        BETA = r.slope if lo > 0 else None
        BETA_NOTE = f"β = {r.slope:.2f} (95% CI {lo:.2f}–{hi:.2f}, {len(fitd)} DUNs)" if BETA is not None else "no clear link → DUN figures only"
    else:
        BETA, BETA_NOTE = None, f"only {len(fitd)} DUNs with enough sales → DUN figures only"
    print("→", BETA_NOTE)

    fig = go.Figure(go.Scatter(x=np.exp(fitd.G), y=fitd.income_median, mode="markers+text", text=[d.split(" ", 1)[-1] for d in fitd.dun], textposition="top center",
                               marker=dict(size=11, color="#1c5cab"), hovertemplate="%{text}<br>House value RM%{x:,.0f}<br>Median income RM%{y:,.0f}<extra></extra>"))
    if BETA is not None and BETA_OVERRIDE is None:
        xs = np.linspace(fitd.G.min(), fitd.G.max(), 50)
        fig.add_trace(go.Scatter(x=np.exp(xs), y=np.exp(r.intercept + r.slope * xs), mode="lines", line=dict(color="#c3c2b7", width=2), hoverinfo="skip"))
    fig.update_layout(height=480, paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color="#52514e"), showlegend=False, margin=dict(t=90, b=60, l=80, r=30),
                      title=dict(text=f"<b>Richer DUNs have pricier homes</b><br><span style='font-size:13px'>{BETA_NOTE} · log–log fit</span>", x=0.01, xanchor="left"),
                      xaxis=dict(type="log", showgrid=False, title=f"Typical nearby home {VALUE} (RM, NAPIC)", linecolor="#c3c2b7"),
                      yaxis=dict(type="log", showgrid=False, title=f"Median household income {LATEST} (RM, DOSM)", linecolor="#c3c2b7"))
    fig.show()

else:
    BETA, BETA_NOTE, g = None, "no NAPIC files → DUN figures only", pd.DataFrame(columns=["G"])

### Spread each DUN's income inside the DUN
`income_h = DUN median × (V_h / V̄_DUN)^β`, where V̄_DUN is the DUN's population-weighted geometric mean house value, so **each DUN's population-weighted geometric mean equals DOSM's median**. Hexagons without nearby sales keep the DUN median (`ses_source = "DUN only"`).

In [ ]:
H["income_est"] = H.dun_median.astype(float); H["ses_source"] = "DUN only (official)"
if BETA is not None:
    m = H.logV.notna() & H.dun_key.isin(g.index)
    H.loc[m, "income_est"] = H.loc[m, "dun_median"] * np.exp(BETA * (H.loc[m, "logV"] - g.loc[H.loc[m, "dun_key"], "G"].values))
    H.loc[m, "ses_source"] = "DUN × NAPIC house values (estimate)"
print(H.ses_source.value_counts().to_string())

def hexmap(col, cs, title, sub, fmt, label, note, zmin=None, zmax=None):
    v = H[col].astype(float); lo = float(np.nanpercentile(v, 2)) if zmin is None else zmin; hi = float(np.nanpercentile(v, 98)) if zmax is None else zmax
    dl = go.Choroplethmap(geojson=DGEO, locations=D.index, z=np.zeros(len(D)), colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]], showscale=False,
                          marker=dict(line=dict(width=1.2, color="#52514e")), customdata=D.dun, hovertemplate="%{customdata}<extra></extra>")
    return map_fig([legend_bar(cs, lo, hi, label), dl], title, sub, [heat_layer(hex_surface(v.values), cs, lo, hi)], note=note)
lo_, hi_ = H.income_est.quantile([0.03, 0.97])
hexmap("income_est", BLUES, "Estimated median household income",
       f"DUN official median spread by NAPIC house values ({BETA_NOTE}) · ESTIMATE", ",.0f", "RM / month",
       SRC_DUN + " · NAPIC (JPPH)", float(lo_), float(hi_)).show()

ses_source
DUN only (official)    841


## 6 · Cross-check: Meta Relative Wealth Index
Independent of DOSM and NAPIC (built from satellite images and phone connectivity). Tiles are 2.4 km, so only broad agreement is expected. A rank correlation well above 0 means the two pictures agree.

In [ ]:
RWI = None
try:
    if not os.path.exists(RWI_FILE):
        pk = requests.get("https://data.humdata.org/api/3/action/package_show", params={"id": "relative-wealth-index"}, timeout=60).json()["result"]
        res = [r for r in pk["resources"] if re.search(r"\bmys\b|malaysia", (r.get("name", "") + " " + r.get("url", "")).lower())]
        assert res, "Malaysia file not listed in HDX 'relative-wealth-index'"
        print("RWI file:", res[0]["name"])
        open(RWI_FILE, "wb").write(requests.get(res[0]["url"], timeout=300).content)
    RWI = pd.read_csv(RWI_FILE)
except Exception as ex:
    print("RWI not loaded →", ex, "\nDownload Malaysia's file from https://data.humdata.org/dataset/relative-wealth-index and save as", RWI_FILE)
if RWI is not None:
    RWI = RWI[(RWI.latitude.between(BBOX[0], BBOX[2])) & (RWI.longitude.between(BBOX[1], BBOX[3]))]
    rx, ry = to_xy(RWI.latitude, RWI.longitude)
    dd = np.hypot(hx[:, None] - rx[None], hy[:, None] - ry[None])
    H["rwi"] = np.where(dd.min(1) < 1700, RWI.rwi.values[dd.argmin(1)], np.nan)
    for c in ("dun_median", "income_est"):
        k = H.rwi.notna(); rho = stats.spearmanr(H.loc[k, c], H.loc[k, "rwi"]).correlation
        print(f"Spearman rank correlation, RWI vs {c}: {rho:+.2f}  ({k.sum()} hexagons)")
    gd = H.groupby("dun_key").apply(lambda t: np.average(t.rwi.fillna(t.rwi.mean()), weights=t["pop"] + 1e-9)).rename("rwi").to_frame().join(D[["dun", "income_median"]])
    print(f"Across DUNs: Spearman {stats.spearmanr(gd.rwi, gd.income_median).correlation:+.2f}")
    fig = go.Figure(go.Scatter(x=gd.rwi, y=gd.income_median, mode="markers+text", text=[d.split(" ", 1)[-1] for d in gd.dun], textposition="top center",
                               marker=dict(size=11, color="#1c5cab"), hovertemplate="%{text}<br>RWI %{x:.2f}<br>RM%{y:,.0f}<extra></extra>"))
    fig.update_layout(height=460, paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color="#52514e"), margin=dict(t=90, b=60, l=80, r=30),
                      title=dict(text="<b>Does Meta's wealth index agree with DOSM?</b><br><span style='font-size:13px'>Population-weighted RWI per DUN vs official median income</span>", x=0.01, xanchor="left"),
                      xaxis=dict(showgrid=False, zeroline=False, title="Meta Relative Wealth Index (relative, unitless)", linecolor="#c3c2b7"),
                      yaxis=dict(showgrid=False, title=f"Median household income {LATEST} (RM)", linecolor="#c3c2b7"))
    fig.show()

RWI file: Malaysia_relative_wealth_index.csv
Spearman rank correlation, RWI vs dun_median: -0.07  (829 hexagons)
Spearman rank correlation, RWI vs income_est: -0.07  (829 hexagons)
Across DUNs: Spearman -0.17


/tmp/ipykernel_1119/615700389.py:20: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## 7 · Older residents by income band
Ipoh's hexagons are cut into five equal-population income bands (Q1 = lowest estimated income). Note: the 65+ *share* comes from the census areas (mostly one figure for the whole mukim), so this shows **how many** older people live in poorer areas, not whether poorer areas are older.

In [ ]:
o = H.sort_values("income_est"); cw = o["pop"].cumsum() / o["pop"].sum()
H.loc[o.index, "income_band"] = pd.cut(cw, [0, .2, .4, .6, .8, 1.0001], labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])
B = H.groupby("income_band", observed=True).agg(people=("pop", "sum"), aged_65=("p65", "sum"), income_from=("income_est", "min"), income_to=("income_est", "max"))
B["pct_65"] = 100 * B.aged_65 / B.people
display(B.round(0))
fig = go.Figure(go.Bar(x=B.index.astype(str), y=B.aged_65, marker=dict(color=["#1c5cab", "#86b6ef", "#cde2fb", "#cde2fb", "#cde2fb"], cornerradius=4),
                       text=[f"{v:,.0f}" for v in B.aged_65], textposition="outside",
                       customdata=np.stack([B.income_from, B.income_to], -1), hovertemplate="%{x}: %{y:,.0f} aged 65+<br>RM%{customdata[0]:,.0f}–%{customdata[1]:,.0f}<extra></extra>"))
fig.update_layout(height=420, paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color="#52514e"), margin=dict(t=90, b=50, l=60, r=30),
                  title=dict(text=f"<b>{B.aged_65.iloc[:2].sum():,.0f} residents aged 65+ live in Ipoh's lowest-income 40% of areas</b><br><span style='font-size:13px'>Five equal-population bands of estimated household income · {BETA_NOTE}</span>", x=0.01, xanchor="left"),
                  xaxis=dict(showgrid=False, linecolor="#c3c2b7"), yaxis=dict(showgrid=False, visible=False))
fig.show()

H["p65_low40"] = np.where(H.income_band.isin(["Q1 lowest", "Q2"]), H.p65, 0.0)
hexmap("p65_low40", [[0, "#fff4ec"], [0.3, "#fde0cc"], [0.55, "#f8b48c"], [0.8, "#b9461a"], [1, "#6e2408"]],
       "Where older residents live in lower-income areas", "Residents aged 65+ in hexagons of the two lowest income bands (others shown blank) · ESTIMATE",
       ",.0f", "Aged 65+", SRC_DUN + " · NAPIC (JPPH) · population: ipoh_pop65_by_hex.csv", 0, float(H.p65_low40.quantile(0.98)) or 1).show()

,people,aged_65,income_from,income_to,pct_65
income_band,,,,,
Q1 lowest,152918.0,15100.0,3165.0,4434.0,10.0
Q2,156085.0,14833.0,4434.0,4988.0,10.0
Q3,153213.0,14096.0,5145.0,5528.0,9.0
Q4,155208.0,14258.0,5528.0,5843.0,9.0
Q5 highest,155103.0,15921.0,5843.0,7335.0,10.0


In [ ]:
cols = ["hex_id", "lat", "lon", "dun", "dun_median", "eff_sales", "income_est", "ses_source", "income_band", "pop", "p65"] + (["rwi"] if "rwi" in H else [])
H[cols].round(2).to_csv("outputs/ipoh_ses_by_hex.csv", index=False)
D.reset_index().round(1).to_csv("outputs/ipoh_ses_by_dun.csv", index=False)
if NAPIC_OK: S.to_csv("outputs/napic_schemes_kinta.csv", index=False)
print("Saved outputs/ipoh_ses_by_hex.csv, ipoh_ses_by_dun.csv, napic_schemes_kinta.csv, napic_unmatched_schemes.csv")

Saved outputs/ipoh_ses_by_hex.csv, ipoh_ses_by_dun.csv, napic_schemes_kinta.csv, napic_unmatched_schemes.csv


### 2b · Poverty rate (official, district level)
OpenDOSM publishes household poverty by district (`hh_poverty_district`), state and nationally, from the same income survey. Kinta is the finest level available — it cannot vary inside Ipoh, so it is shown as context and **not** used in the area index.

In [ ]:
def rd_pq(u):
    d = pd.read_parquet(u) if u.endswith(".parquet") else pd.read_csv(u)
    if "date" in d: d["date"] = pd.to_datetime(d["date"])
    return d
POV = {"Kinta district": ("https://storage.dosm.gov.my/hies/hh_poverty_district.parquet", "district", "kinta"),
       "Perak state":    ("https://storage.dosm.gov.my/hies/hh_poverty_state.parquet", "state", "perak"),
       "Malaysia":       ("https://storage.dosm.gov.my/hies/hh_poverty.parquet", None, None)}
prow = []
for label, (u, col, val) in POV.items():
    try:
        d = rd_pq(u)
        if col: d = d[d[col].str.lower() == val]
        d = d.sort_values("date").iloc[[-1]]
        r = {"area": label, "year": int(d.date.dt.year.iloc[0])}
        r.update({c: float(d[c].iloc[0]) for c in d.columns if "poverty" in c.lower()})
        prow.append(r)
    except Exception as ex:
        print(f"⚠ {label}: could not load {u} → {ex}")
POVERTY = pd.DataFrame(prow).set_index("area") if prow else pd.DataFrame()
print("Household poverty rate, % of households (DOSM HIES):"); display(POVERTY)

Household poverty rate, % of households (DOSM HIES):


,year,poverty_absolute,poverty_relative,poverty_hardcore
area,,,,
Kinta district,2024,3.8,10.0,NaN
Perak state,2024,6.9,11.4,0.06
Malaysia,2024,5.1,16.7,0.09


# PART 4 · CARE — how far are care homes and senior activity centres?
**Data:**
* **JKM's PAWE list** (Pusat Aktiviti Warga Emas, senior activity centres): official members/attendance by state, parliament seat and DUN, 2019–2021, from data.gov.my's archive. Centres are placed by name with OpenStreetMap's address search (Nominatim), and every placement is printed.
* **OpenStreetMap** care homes and social facilities for older people.
* **Not public:** JKM's register of licensed care homes and their capacity. Request it from JKM; until then, the *number of places* can't be measured.

In [ ]:
import time, io
from scipy.sparse import coo_matrix as _coo
H = pd.read_csv(HEX_CSV)                                   # fresh copy of the hexagons (same order as Part 2)
assert np.allclose(to_xy(H.lat, H.lon)[0], HX_HEALTH)
def _virtual_dijkstra(Gm, nodes, offsets, n_nodes, limit=None):
    s = pd.Series(offsets, index=nodes).groupby(level=0).min() + 1e-3
    Gc = Gm.tocoo()
    E = _coo((np.r_[Gc.data, s.values], (np.r_[Gc.row, np.full(len(s), n_nodes)], np.r_[Gc.col, s.index.values])),
             shape=(n_nodes + 1, n_nodes + 1)).tocsr()
    return dijkstra(E, directed=True, indices=n_nodes, limit=limit if limit else np.inf)[:n_nodes]
def walk_from(lat, lon, max_snap=400):
    """Walking minutes from every street node (and every hexagon) to the nearest of these points."""
    x, y = to_xy(lat, lon); n, sd = snap(x, y); ok = sd <= max_snap
    if not ok.any(): return None, np.full(len(H), np.nan)
    m = _virtual_dijkstra(G, n[ok], sd[ok], NN, 40_000)
    return m / WALK_MPS / 60, (m[H_node] + H_snap) / WALK_MPS / 60
def drive_from(lat, lon, max_snap=400):
    """Driving minutes (to the point, one-way streets respected) from every road node and hexagon."""
    x, y = to_xy(lat, lon); n, sd = dsnap(x, y); ok = sd <= max_snap
    if not ok.any(): return None, np.full(len(H), np.nan)
    m = _virtual_dijkstra(GDT, n[ok], np.zeros(ok.sum()), DN) / 60 + ACCESS_MIN
    return m, m[H_dnode] + H_dsnap / WALK_MPS / 60

# ---- OSM care facilities --------------------------------------------------------------------------------
q = (f'[out:json][timeout:300];(nwr["amenity"~"^(nursing_home|social_facility)$"]({bb});'
     f'nwr["social_facility"~"nursing_home|assisted_living|group_home|day_care"]({bb});'
     f'nwr["name"~"warga emas|orang tua|old folk|senior citizen|pawe|rumah jagaan|pusat jagaan|elderly|nursing home",i]({bb}););out center tags;')
CARE = []
for el in overpass(q, "data/osm_ipoh_care.json").get("elements", []):
    t = el.get("tags", {}); la = el.get("lat", el.get("center", {}).get("lat")); lo = el.get("lon", el.get("center", {}).get("lon"))
    nm_ = t.get("name", ""); low = nm_.lower()
    if la is None or re.search(r"kanak|taska|tadika|child|anak yatim|orphan|kindergarten|daycare kids|nurseri", low): continue
    sf_for = t.get("social_facility:for", "")
    if t.get("amenity") == "social_facility" and sf_for and not re.search(r"senior|elder|aged", sf_for) and not re.search(r"warga emas|orang tua|old|senior|elder", low): continue
    kind = "Senior activity centre (PAWE)" if re.search(r"pawe|aktiviti warga emas", low) else "Care home / nursing home"
    CARE.append(dict(src="OSM", name=nm_ or "(no name)", lat=la, lon=lo, kind=kind))
CARE = pd.DataFrame(CARE, columns=["src", "name", "lat", "lon", "kind"])
print(f"OSM care facilities for older people near Ipoh: {len(CARE)}"); display(CARE.groupby("kind").size().rename("count").to_frame())

# ---- JKM PAWE list (official) ---------------------------------------------------------------------------
PAWE_RES = "f2d4422e-e32e-446e-bcdb-c8718ebc47b7"
PAWE_FILE = "data/jkm_pawe_members.csv"; PAWE = None
try:
    if not os.path.exists(PAWE_FILE):
        for u in (f"https://archive.data.gov.my/data/datastore/dump/{PAWE_RES}", f"https://archive.data.gov.my/data/en_US/datastore/dump/{PAWE_RES}"):
            r = requests.get(u, timeout=60)
            if r.status_code == 200 and len(r.content) > 200: open(PAWE_FILE, "wb").write(r.content); break
    PAWE = pd.read_csv(PAWE_FILE)
    lc_ = {c.lower(): c for c in PAWE.columns}
    stc = next(c for k, c in lc_.items() if "negeri" in k or "state" in k)
    PAWE = PAWE[PAWE[stc].astype(str).str.lower().str.contains("perak")]
    print(f"JKM PAWE list: {len(PAWE)} Perak rows · columns {list(PAWE.columns)}")
    display(PAWE.head(30))
except Exception as ex:
    print("⚠ JKM PAWE list not loaded →", ex, f"\n  Download it from archive.data.gov.my (resource {PAWE_RES}) and save as {PAWE_FILE}.")
if PAWE is not None and len(PAWE):
    namec = next((c for k, c in lc_.items() if "pawe" in k or "nama" in k or "name" in k), None)
    dunc = next((c for k, c in lc_.items() if k == "dun" or "dun" in k), None)
    geo_f = "data/nominatim_pawe.json"; geo = json.load(open(geo_f)) if os.path.exists(geo_f) else {}
    have = " | ".join(CARE.name.str.lower())
    for nm_ in PAWE[namec].dropna().astype(str).unique():
        if nm_.lower() in have: continue
        if nm_ not in geo:
            for qq in (f"PAWE {nm_}, Perak", f"{nm_}, Perak"):
                try:
                    rr = requests.get("https://nominatim.openstreetmap.org/search", params={"q": qq, "format": "json", "limit": 1, "countrycodes": "my",
                                      "viewbox": f"{BBOX[1]},{BBOX[2]},{BBOX[3]},{BBOX[0]}", "bounded": 1}, headers={"User-Agent": "ipoh-ageing/1.0"}, timeout=30).json()
                except Exception: rr = []
                time.sleep(1.1)
                if rr: geo[nm_] = [float(rr[0]["lat"]), float(rr[0]["lon"]), qq]; break
            geo.setdefault(nm_, None); json.dump(geo, open(geo_f, "w"))
        if geo.get(nm_):
            CARE.loc[len(CARE)] = ["JKM PAWE list (geocoded)", f"PAWE {nm_}", geo[nm_][0], geo[nm_][1], "Senior activity centre (PAWE)"]
            print(f"  + PAWE {nm_}: placed from '{geo[nm_][2]}'")
        else:
            print(f"  ⚠ PAWE {nm_}: not found inside the Ipoh area (outside Ipoh, or add it by hand)")

# ---- travel times to care -------------------------------------------------------------------------------
CARE_NODE_W, CARE_NODE_D = {}, {}
for kind in ["Care home / nursing home", "Senior activity centre (PAWE)"]:
    s = CARE[CARE.kind == kind]
    if not len(s): print("  none found:", kind); H["walk_" + kind] = np.nan; H["drive_" + kind] = np.nan; continue
    CARE_NODE_W[kind], H["walk_" + kind] = walk_from(s.lat, s.lon)
    CARE_NODE_D[kind], H["drive_" + kind] = drive_from(s.lat, s.lon)
for kind in CARE_NODE_W:
    w = H["walk_" + kind]; d = H["drive_" + kind]
    print(f"{kind}: {100 * H.p65[w <= 15].sum() / H.p65.sum():.0f}% of 65+ within a 15-min walk · {100 * H.p65[d <= 10].sum() / H.p65.sum():.0f}% within a 10-min drive")

def care_traces(kinds):
    out = []
    for kind, col in [("Care home / nursing home", "#6e2408"), ("Senior activity centre (PAWE)", "#1c5cab")]:
        s = CARE[CARE.kind == kind]
        if kind in kinds and len(s):
            out.append(go.Scattermap(lat=s.lat, lon=s.lon, mode="markers", name=f"{kind} ({len(s)})", marker=dict(size=12, color=col),
                                     text=s.name + " · " + s.src, hovertemplate="%{text}<extra></extra>"))
    return out
for kind in CARE_NODE_W:
    map_fig([legend_bar(TIME_CS, 0, 60, "Minutes")] + care_traces([kind]), f"Walking time to the nearest {kind.lower()}",
            f"Along real streets at {WALK_MPS} m/s · lines = 15 / 30 / 45 min · OSM + JKM list (OSM is incomplete for care homes)",
            [heat_layer(surface(CARE_NODE_W[kind]), TIME_CS, 0, 60, [15, 30, 45])]).show()
H[["hex_id"] + [c for c in H.columns if c.startswith(("walk_Care", "walk_Senior", "drive_Care", "drive_Senior"))]].round(2).to_csv("outputs/ipoh_care_by_hex.csv", index=False)
CARE.to_csv("outputs/ipoh_care_facilities.csv", index=False)

   busy/failed: HTTP 502
   busy/failed: HTTP 502
OSM care facilities for older people near Ipoh: 18


,count
kind,
Care home / nursing home,18


⚠ JKM PAWE list not loaded → HTTPSConnectionPool(host='archive.data.gov.my', port=443): Max retries exceeded with url: /data/datastore/dump/f2d4422e-e32e-446e-bcdb-c8718ebc47b7 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7e16808b34d0>, 'Connection to archive.data.gov.my timed out. (connect timeout=60)')) 
  Download it from archive.data.gov.my (resource f2d4422e-e32e-446e-bcdb-c8718ebc47b7) and save as data/jkm_pawe_members.csv.
  none found: Senior activity centre (PAWE)
Care home / nursing home: 6% of 65+ within a 15-min walk · 57% within a 10-min drive


# PART 5 · SOCIAL — places to meet within a 10-minute walk
Counts real places reachable **along streets** within 10 minutes at an older person's pace (about 540 m), from OpenStreetMap. There are five kinds: places of worship, community halls and libraries, parks, kopitiams and food courts, and markets.
The index uses **how many of the five kinds are within reach** (0–5). That's sturdier than raw counts, which mostly reflect how thoroughly volunteers have mapped an area.

In [ ]:
q = (f'[out:json][timeout:300];(nwr["amenity"~"^(place_of_worship|community_centre|social_centre|library|cafe|restaurant|food_court|marketplace)$"]({bb});'
     f'nwr["leisure"~"^(park|garden|fitness_station|pitch|sports_centre)$"]["access"!="private"]({bb});nwr["shop"="supermarket"]({bb}););out center tags;')
SOC = []
for el in overpass(q, "data/osm_ipoh_social.json").get("elements", []):
    t = el.get("tags", {}); la = el.get("lat", el.get("center", {}).get("lat")); lo = el.get("lon", el.get("center", {}).get("lon"))
    if la is None: continue
    a, l_ = t.get("amenity", ""), t.get("leisure", "")
    k = ("Worship" if a == "place_of_worship" else "Community hall / library" if a in ("community_centre", "social_centre", "library")
         else "Park / exercise" if l_ else "Kopitiam / food court" if a in ("cafe", "restaurant", "food_court") else "Market")
    SOC.append((la, lo, k, t.get("name", "")))
SOC = pd.DataFrame(SOC, columns=["lat", "lon", "kind", "name"])
SOC_KINDS = ["Worship", "Community hall / library", "Park / exercise", "Kopitiam / food court", "Market"]
print("OSM meeting places:"); display(SOC.groupby("kind").size().reindex(SOC_KINDS).rename("count").to_frame())
sx_, sy_ = to_xy(SOC.lat, SOC.lon); SOC["node"], SOC["snap"] = snap(sx_, sy_)
SOC = SOC[SOC.snap <= 150].reset_index(drop=True)
LIM = WALK_MPS * 600                                           # metres in 10 minutes
CNT = np.zeros((len(H), len(SOC_KINDS)), int)
kcode = SOC.kind.map({k: i for i, k in enumerate(SOC_KINDS)}).values
for i0 in range(0, len(H), 40):                                 # in chunks to keep memory small
    idx = np.arange(i0, min(i0 + 40, len(H)))
    Dm = dijkstra(G, directed=False, indices=H_node[idx], limit=LIM)[:, SOC.node.values] + SOC.snap.values[None] + H_snap[idx, None]
    ok = Dm <= LIM
    for j in range(len(SOC_KINDS)): CNT[idx, j] = (ok & (kcode == j)[None]).sum(1)
for j, k in enumerate(SOC_KINDS): H["soc_" + k] = CNT[:, j]
H["soc_total"] = CNT.sum(1); H["soc_kinds"] = (CNT > 0).sum(1)
print(f"Residents aged 65+ with ≥3 of the 5 kinds within a 10-min walk: {100 * H.p65[H.soc_kinds >= 3].sum() / H.p65.sum():.0f}% · "
      f"with none: {100 * H.p65[H.soc_kinds == 0].sum() / H.p65.sum():.0f}%")
H[["hex_id", "soc_total", "soc_kinds"] + ["soc_" + k for k in SOC_KINDS]].to_csv("outputs/ipoh_social_by_hex.csv", index=False)

   busy/failed: HTTP 502
   busy/failed: HTTP 504
OSM meeting places:


,count
kind,
Worship,352
Community hall / library,55
Park / exercise,738
Kopitiam / food court,754
Market,171


Residents aged 65+ with ≥3 of the 5 kinds within a 10-min walk: 19% · with none: 39%


# PART 6 · ACCESS — buses, housing, internet
### 6a · Bus stops within a 400 m walk
The UN SDG 11.2.1 indicator counts people with "convenient access" as living within **500 m** of a bus stop. Here a stricter **400 m along streets** is used for older people, with 500 m reported alongside it. Bus stops come from OpenStreetMap; frequency comes from the official BAS.MY Ipoh timetable (GTFS) when data.gov.my serves it.

In [ ]:
q = f'[out:json][timeout:300];(node["highway"="bus_stop"]({bb});node["public_transport"="platform"]["bus"="yes"]({bb}););out;'
STOPS = pd.DataFrame([(e["lat"], e["lon"], e.get("tags", {}).get("name", "")) for e in overpass(q, "data/osm_ipoh_busstops.json").get("elements", []) if "lat" in e],
                     columns=["lat", "lon", "name"]).drop_duplicates(["lat", "lon"])
print(f"OSM bus stops: {len(STOPS):,}")
BUS_NODE_W, H["walk_bus_stop"] = walk_from(STOPS.lat, STOPS.lon) if len(STOPS) else (None, np.nan)
H["m_bus_stop"] = H.walk_bus_stop * WALK_MPS * 60
for d_ in (400, 500):
    print(f"  {100 * H.p65[H.m_bus_stop <= d_].sum() / H.p65.sum():.0f}% of residents aged 65+ live within {d_} m (along streets) of a bus stop")

# ---- official timetable → buses per hour (weekday 07:00–19:00) ------------------------------------------
GTFS = None
try:
    r = requests.get("https://api.data.gov.my/gtfs-static/mybas-ipoh", timeout=120)
    if r.status_code == 200 and r.content[:2] == b"PK":
        import zipfile as _zf
        z = _zf.ZipFile(io.BytesIO(r.content)); rd_ = lambda f: pd.read_csv(z.open(f), dtype=str)
        stops_g, trips_g, st_g = rd_("stops.txt"), rd_("trips.txt"), rd_("stop_times.txt")
        if "calendar.txt" in z.namelist():
            cal = rd_("calendar.txt"); svc = set(cal.loc[cal.get("wednesday", "0") == "1", "service_id"])
        else:
            svc = set(trips_g.service_id); print("  (no calendar.txt → all trips counted)")
        st_g = st_g[st_g.trip_id.isin(trips_g.loc[trips_g.service_id.isin(svc), "trip_id"])]
        hh = pd.to_numeric(st_g.departure_time.str.split(":").str[0], errors="coerce")
        f = st_g[(hh >= 7) & (hh < 19)].groupby("stop_id").size() / 12.0
        GTFS = stops_g.assign(lat=stops_g.stop_lat.astype(float), lon=stops_g.stop_lon.astype(float), per_hour=stops_g.stop_id.map(f).fillna(0))
        print(f"BAS.MY Ipoh timetable: {len(GTFS)} stops, {st_g.trip_id.nunique()} weekday trips")
    else:
        print(f"⚠ BAS.MY Ipoh timetable not served (HTTP {r.status_code}) → bus FREQUENCY not measured (stop distance still is).")
except Exception as ex:
    print("⚠ BAS.MY Ipoh timetable not loaded →", ex)
if GTFS is not None:
    for thr in (1, 2, 4):
        g_ = GTFS[GTFS.per_hour >= thr]
        H[f"walk_stop_{thr}ph"] = walk_from(g_.lat, g_.lon)[1] if len(g_) else np.nan
        print(f"  {100 * H.p65[H[f'walk_stop_{thr}ph'] * WALK_MPS * 60 <= 400].sum() / H.p65.sum():.0f}% of 65+ within 400 m of a stop with ≥{thr} bus/hour")
if BUS_NODE_W is not None:
    stop_tr = [go.Scattermap(lat=STOPS.lat, lon=STOPS.lon, mode="markers", name=f"Bus stop ({len(STOPS)})", marker=dict(size=5, color="#14171C"), hoverinfo="skip")]
    map_fig([legend_bar(TIME_CS, 0, 20, "Minutes")] + stop_tr, "Walking time to the nearest bus stop",
            f"Along real streets at {WALK_MPS} m/s · line = 400 m (≈ {400 / WALK_MPS / 60:.0f} min) · stops: OpenStreetMap",
            [heat_layer(surface(BUS_NODE_W), TIME_CS, 0, 20, [400 / WALK_MPS / 60])]).show()

OSM bus stops: 378
  16% of residents aged 65+ live within 400 m (along streets) of a bus stop
  23% of residents aged 65+ live within 500 m (along streets) of a bus stop
BAS.MY Ipoh timetable: 2079 stops, 369 weekday trips
  18% of 65+ within 400 m of a stop with ≥1 bus/hour
  1% of 65+ within 400 m of a stop with ≥2 bus/hour
  0% of 65+ within 400 m of a stop with ≥4 bus/hour


### 6b · Housing: flats among nearby home sales (NAPIC)
Older people in walk-up flats struggle with stairs. NAPIC records each sale's building type, so this measures **the share of nearby recorded sales that are flats** (Flat / Low-Cost Flat). It's a proxy for how common flats are in the area; it doesn't count every home. It needs `outputs/napic_schemes_kinta.csv` from Part 2.

In [ ]:
NS = "outputs/napic_schemes_kinta.csv"
if os.path.exists(NS):
    S_ = pd.read_csv(NS).dropna(subset=["lat", "lon"])
    flat = S_.type.astype(str).str.contains(r"\bflat\b", case=False)
    ax_, ay_ = to_xy(S_.lat, S_.lon)
    d2_ = (hx[:, None] - ax_[None]) ** 2 + (hy[:, None] - ay_[None]) ** 2
    Wk = S_.n.values[None] * np.exp(-0.5 * d2_ / 600 ** 2) * (d2_ < 1800 ** 2)
    H["flat_share"] = np.where(Wk.sum(1) >= 10, 100 * (Wk * flat.values[None]).sum(1) / np.maximum(Wk.sum(1), 1e-9), np.nan)
    print(f"Flats: {flat.sum()} of {len(S_)} placed tamans are flats · {H.flat_share.notna().mean():.0%} of hexagons have enough nearby sales")
else:
    H["flat_share"] = np.nan; print(f"⚠ {NS} missing (run Part 2 with NAPIC files) → housing type not measured.")

⚠ outputs/napic_schemes_kinta.csv missing (run Part 2 with NAPIC files) → housing type not measured.


### 6c · Internet speed (Ookla open data)
Speedtest by Ookla publishes median speeds for 610 m tiles every quarter. **The licence is CC BY-NC-SA 4.0, which is non-commercial only.** Fine for research and public-interest work; for a paid consultancy deliverable, ask Ookla (opendata@ookla.com) first. Set `OOKLA = False` to skip it.
The files are global (a few hundred MB per quarter). They're downloaded once to `data/`.

In [ ]:
OOKLA = True
def tile_xy(lat, lon, z):
    n = 2 ** z; x = int((lon + 180) / 360 * n); y = int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n); return x, y
def quadkey(x, y, z): return "".join(str(((x >> (z - i - 1)) & 1) + 2 * ((y >> (z - i - 1)) & 1)) for i in range(z))
def qk_center(qk):
    z = len(qk); x = y = 0
    for i, c in enumerate(qk):
        m = 1 << (z - i - 1); d = int(c); x |= m if d & 1 else 0; y |= m if d & 2 else 0
    n = 2 ** z; lon = (x + 0.5) / n * 360 - 180; lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 0.5) / n)))); return lat, lon
def ookla_tiles(kind):
    import datetime as _dt
    y_, q_ = _dt.date.today().year, (_dt.date.today().month - 1) // 3 + 1
    for _ in range(10):
        q_ -= 1
        if q_ == 0: y_, q_ = y_ - 1, 4
        mon = {1: "01", 2: "04", 3: "07", 4: "10"}[q_]
        fn = f"data/ookla_{kind}_{y_}q{q_}.parquet"
        url = f"https://ookla-open-data.s3.amazonaws.com/parquet/performance/type={kind}/year={y_}/quarter={q_}/{y_}-{mon}-01_performance_{kind}_tiles.parquet"
        if not os.path.exists(fn):
            if requests.head(url, timeout=30).status_code != 200: continue
            print(f"Downloading Ookla {kind} {y_} Q{q_} …")
            with requests.get(url, stream=True, timeout=1800) as rr:
                with open(fn, "wb") as fh:
                    for ch in rr.iter_content(1 << 22): fh.write(ch)
        x0, y0 = tile_xy(BBOX[2], BBOX[1], 10); x1, y1 = tile_xy(BBOX[0], BBOX[3], 10)
        pref = {quadkey(x, y, 10) for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)}
        T = pd.read_parquet(fn, columns=["quadkey", "avg_d_kbps", "avg_u_kbps", "tests", "devices"])
        T = T[T.quadkey.str[:10].isin(pref)].copy()
        T[["lat", "lon"]] = [qk_center(q) for q in T.quadkey]
        return T, f"{y_} Q{q_}"
    return None, None
if OOKLA:
    for kind in ("fixed", "mobile"):
        try:
            T, per = ookla_tiles(kind)
            if T is None: print(f"⚠ no Ookla {kind} file found"); continue
            tx, ty = to_xy(T.lat, T.lon); dd_, ii_ = cKDTree(np.c_[tx, ty]).query(np.c_[hx, hy])
            H[f"net_{kind}_mbps"] = np.where(dd_ <= 450, T.avg_d_kbps.values[ii_] / 1000, np.nan)
            print(f"Ookla {kind} {per}: {len(T)} tiles near Ipoh · median download {np.nanmedian(H[f'net_{kind}_mbps']):.0f} Mbps · "
                  f"{H[f'net_{kind}_mbps'].notna().mean():.0%} of hexagons have a tile with tests")
        except Exception as ex:
            print(f"⚠ Ookla {kind} not loaded →", ex)
H[["hex_id", "walk_bus_stop", "m_bus_stop", "flat_share"] + [c for c in H.columns if c.startswith(("walk_stop_", "net_"))]].round(2).to_csv("outputs/ipoh_access_by_hex.csv", index=False)

Ookla fixed 2026 Q3: 1469 tiles near Ipoh · median download 244 Mbps · 80% of hexagons have a tile with tests
Ookla mobile 2026 Q3: 1954 tiles near Ipoh · median download 222 Mbps · 82% of hexagons have a tile with tests


# PART 7 · THE AGEING-VULNERABILITY INDEX
**How it's built**
1. Every metric is turned into a **0–100 score where higher = worse for an older person**, using its percentile rank across Ipoh's hexagons. Ranks aren't thrown off by a few extreme values.
2. **Domain score** = the average of that domain's available metrics. Metrics missing for a hexagon are skipped, not filled in.
3. **Overall index** = the average of the five domains. A **Monte Carlo test** then re-weights the domains randomly 3,000 times. An area that stays in the worst 10% under most weightings is a robust priority, not an artefact of the chosen weights.

**Read this before using it:** Care and Social are measured here only by *distance to places*. How people are actually doing (illness, loneliness, who cares for them) needs the survey in Part 8.

In [ ]:
M = pd.read_csv(HEX_CSV)[["hex_id", "lat", "lon", "pop", "p65"]]
for f in ["outputs/ipoh_ses_by_hex.csv", "outputs/ipoh_health_access_by_hex.csv", "outputs/ipoh_care_by_hex.csv",
          "outputs/ipoh_social_by_hex.csv", "outputs/ipoh_access_by_hex.csv"]:
    if os.path.exists(f):
        X = pd.read_csv(f); X = X[[c for c in X.columns if c == "hex_id" or c not in M.columns]]
        M = M.merge(X, on="hex_id", how="left")
    else:
        print("⚠ missing (that part was skipped):", f)
# metric → (domain, direction: +1 = higher value is WORSE, -1 = higher value is BETTER, label)
SPEC = {"income_est": ("Money", -1, "Estimated household income"),
        "min_Public clinic (KKM)": ("Health", +1, "Walk to public clinic"),
        "drive_Public clinic (KKM)": ("Health", +1, "Drive to public clinic"),
        "drive_Hospital": ("Health", +1, "Drive to hospital"),
        "clinics_per_10k": ("Health", -1, "Public-clinic capacity"),
        "walk_Care home / nursing home": ("Care", +1, "Walk to care home"),
        "drive_Care home / nursing home": ("Care", +1, "Drive to care home"),
        "walk_Senior activity centre (PAWE)": ("Care", +1, "Walk to senior activity centre"),
        "soc_kinds": ("Social", -1, "Kinds of meeting place within 10 min"),
        "walk_bus_stop": ("Access", +1, "Walk to bus stop"),
        "walk_stop_2ph": ("Access", +1, "Walk to stop with ≥2 buses/hour"),
        "flat_share": ("Access", +1, "Share of flats among nearby sales"),
        "net_mobile_mbps": ("Access", -1, "Mobile internet speed"),
        "net_fixed_mbps": ("Access", -1, "Fixed internet speed")}
DOMAINS = ["Money", "Health", "Care", "Social", "Access"]
used = []
for c, (dom, sgn, lab) in SPEC.items():
    if c in M and M[c].notna().mean() >= 0.3 and M[c].nunique() > 2:
        r = M[c].rank(pct=True) * 100
        M["s_" + c] = r if sgn > 0 else 100 - r + (100 / M[c].notna().sum())
        used.append((dom, lab, c, M[c].notna().mean()))
    elif c in M:
        print(f"  not used (too few values): {lab}")
U = pd.DataFrame(used, columns=["domain", "metric", "column", "coverage"])
print("Metrics in the index:"); display(U.assign(coverage=(100 * U.coverage).round(0).astype(int).astype(str) + "%"))
for d in DOMAINS:
    cols = ["s_" + c for c in U.loc[U.domain == d, "column"]]
    M["D_" + d] = M[cols].mean(axis=1) if cols else np.nan
DOM_OK = [d for d in DOMAINS if M["D_" + d].notna().any()]
M["index"] = M[["D_" + d for d in DOM_OK]].mean(axis=1)

rng = np.random.default_rng(7); Wd = rng.dirichlet(np.ones(len(DOM_OK)), 3000)
Dmat = M[["D_" + d for d in DOM_OK]].fillna(M[["D_" + d for d in DOM_OK]].mean()).values
scores = Dmat @ Wd.T
top10 = scores >= np.quantile(scores, 0.9, axis=0)[None]
M["p_top10"] = 100 * top10.mean(1)
print(f"Hexagons in the worst 10% under ≥80% of 3,000 random weightings: {(M.p_top10 >= 80).sum()} "
      f"({M.p65[M.p_top10 >= 80].sum():,.0f} residents aged 65+)")

# ---- maps: smooth surfaces of domain scores and the index ----------------------------------------------
IDX_CS = [[0, "#f2f7f2"], [0.25, "#d9ecd0"], [0.5, "#fdd49e"], [0.75, "#e34a33"], [1, "#7f0000"]]
assert (M.hex_id.values == H.hex_id.values).all()
map_fig([legend_bar(IDX_CS, 0, 100, "Score")], "Ageing-vulnerability index, Ipoh",
        f"Average of {len(DOM_OK)} domain scores ({', '.join(DOM_OK)}) · 0 = best-off area, 100 = worst-off · line = worst 10% under most weightings",
        [heat_layer(hex_surface(M["index"]), IDX_CS, 0, 100)]).show()
for d in DOM_OK:
    labs = ", ".join(U.loc[U.domain == d, "metric"])
    map_fig([legend_bar(IDX_CS, 0, 100, "Score")], f"{d} domain score", f"From: {labs} · higher = worse for older residents",
            [heat_layer(hex_surface(M["D_" + d]), IDX_CS, 0, 100)]).show()

# ---- who is affected: 65+ by index band, and by DUN -------------------------------------------------------
o = M.sort_values("index"); cw = o.p65.cumsum() / o.p65.sum()
M.loc[o.index, "band"] = pd.cut(cw, [0, .2, .4, .6, .8, 1.0001], labels=["Q1 best-off", "Q2", "Q3", "Q4", "Q5 worst-off"])
B5 = M.groupby("band", observed=True).agg(aged_65=("p65", "sum"), people=("pop", "sum"), **{d: ("D_" + d, "mean") for d in DOM_OK})
display(B5.round(0))
fig = go.Figure()
DCOL = {"Money": "#1c5cab", "Health": "#d7301f", "Care": "#9b6bcc", "Social": "#3a9a5b", "Access": "#c9a227"}
for d in DOM_OK:
    fig.add_trace(go.Bar(x=B5.index.astype(str), y=B5[d], name=d, marker=dict(color=DCOL[d], cornerradius=3, line=dict(color="white", width=2)),
                         hovertemplate="%{x}: " + d + " score %{y:.0f}<extra></extra>"))
fig.update_layout(barmode="group", height=440, paper_bgcolor="white", plot_bgcolor="white", font=dict(family=FONT, color=INK),
                  margin=dict(t=100, b=50, l=60, r=30), legend=dict(orientation="h", y=1.04, x=0),
                  title=dict(text="<b>What drives vulnerability in each band</b><br><span style='font-size:13px'>Average domain score by index band (each band holds 20% of residents aged 65+)</span>", x=0.01, xanchor="left"),
                  xaxis=dict(showgrid=False, linecolor="#c3c2b7"), yaxis=dict(showgrid=False, title="Score (higher = worse)", range=[0, 100]))
fig.show()
if "dun" in M:
    DT = M.groupby("dun").apply(lambda t: pd.Series({"aged_65": t.p65.sum(), **{d: np.average(t["D_" + d].fillna(t["D_" + d].mean()), weights=t.p65 + 1e-9) for d in DOM_OK},
                                                       "index": np.average(t["index"], weights=t.p65 + 1e-9)})).sort_values("index", ascending=False)
    print("By state constituency (DUN), weighted by residents aged 65+ — higher = worse:"); display(DT.round(0))

# ---- priority list with a recognisable place name -------------------------------------------------------
NAMES = []
for f in ["data/osm_ipoh_names.json", OSM_CACHE]:
    if os.path.exists(f):
        for el in json.load(open(f, encoding="utf-8")).get("elements", []):
            t = el.get("tags", {}); la = el.get("lat", el.get("center", {}).get("lat")); lo = el.get("lon", el.get("center", {}).get("lon"))
            if la is not None and t.get("name") and (t.get("place") or t.get("landuse") == "residential"): NAMES.append((la, lo, t["name"]))
if NAMES:
    Nn = pd.DataFrame(NAMES, columns=["lat", "lon", "name"]); nx_, ny_ = to_xy(Nn.lat, Nn.lon)
    M["near_place"] = Nn.name.values[cKDTree(np.c_[nx_, ny_]).query(np.c_[hx, hy])[1]]
M["priority"] = M.p65 * M["index"] / 100
cols = [c for c in ["hex_id", "near_place", "dun", "p65", "index", "p_top10"] + ["D_" + d for d in DOM_OK] if c in M]
print("Top 20 priority areas (residents aged 65+ × index):"); display(M.sort_values("priority", ascending=False)[cols].head(20).round(1))
M.round(3).to_csv("outputs/ipoh_ageing_index_by_hex.csv", index=False)
print("Saved outputs/ipoh_ageing_index_by_hex.csv")

  not used (too few values): Walk to senior activity centre
  not used (too few values): Share of flats among nearby sales
Metrics in the index:


,domain,metric,column,coverage
0,Money,Estimated household income,income_est,100%
1,Health,Walk to public clinic,min_Public clinic (KKM),100%
2,Health,Drive to public clinic,drive_Public clinic (KKM),100%
3,Health,Drive to hospital,drive_Hospital,100%
4,Health,Public-clinic capacity,clinics_per_10k,100%
5,Care,Walk to care home,walk_Care home / nursing home,100%
6,Care,Drive to care home,drive_Care home / nursing home,100%
7,Social,Kinds of meeting place within 10 min,soc_kinds,100%
8,Access,Walk to bus stop,walk_bus_stop,100%
9,Access,Walk to stop with ≥2 buses/hour,walk_stop_2ph,100%


Hexagons in the worst 10% under ≥80% of 3,000 random weightings: 28 (153 residents aged 65+)


,aged_65,people,Money,Health,Care,Social,Access
band,,,,,,,
Q1 best-off,14813.0,148896.0,17.0,23.0,11.0,20.0,28.0
Q2,14827.0,148888.0,48.0,26.0,23.0,24.0,36.0
Q3,14835.0,152343.0,61.0,36.0,33.0,33.0,38.0
Q4,14852.0,160665.0,54.0,47.0,50.0,56.0,49.0
Q5 worst-off,14880.0,161734.0,53.0,69.0,72.0,67.0,70.0


By state constituency (DUN), weighted by residents aged 65+ — higher = worse:


/tmp/ipykernel_1119/1533400465.py:76: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,aged_65,Money,Health,Care,Social,Access,index
dun,,,,,,,
N.32 Menglembu,4560.0,95.0,32.0,53.0,45.0,35.0,52.0
N.23 Manjoi,14779.0,60.0,54.0,43.0,40.0,48.0,49.0
N.26 Tebing Tinggi,5400.0,85.0,35.0,46.0,42.0,34.0,48.0
N.28 Bercham,4697.0,90.0,44.0,22.0,41.0,36.0,47.0
N.24 Hulu Kinta,11581.0,33.0,37.0,58.0,49.0,56.0,46.0
N.31 Jelapang,2623.0,79.0,47.0,22.0,44.0,38.0,46.0
N.44 Sungai Rapat,4957.0,74.0,37.0,38.0,33.0,39.0,44.0
N.30 Buntong,2985.0,82.0,23.0,25.0,35.0,27.0,38.0
N.27 Pasir Pinji,4711.0,99.0,16.0,24.0,23.0,29.0,38.0


Top 20 priority areas (residents aged 65+ × index):


,hex_id,near_place,dun,p65,index,p_top10,D_Money,D_Health,D_Care,D_Social,D_Access
369,H-002+008,Taman Loke Lim,N.28 Bercham,225.0,62.9,22.3,90.4,74.7,36.0,69.4,43.9
469,H+001-003,Kampung Paloh,N.27 Pasir Pinji,356.2,38.6,1.2,98.8,10.9,18.7,30.0,34.4
504,H+002-004,Kampung Paloh,N.27 Pasir Pinji,347.6,38.6,1.2,98.8,10.9,18.7,30.0,34.4
505,H+002-003,Kampung Paloh,N.27 Pasir Pinji,347.6,38.6,1.2,98.8,10.9,18.7,30.0,34.4
314,H-003-005,Kg. Sri Kinta,N.32 Menglembu,228.9,55.5,10.1,95.1,33.1,44.1,69.4,35.7
409,H-001+011,Taman Klebang Utama,N.23 Manjoi,191.0,61.9,1.0,59.9,80.1,43.4,69.4,56.6
171,H-007+020,Taman Chemor Idaman,N.23 Manjoi,230.1,51.2,0.0,59.9,62.6,77.7,6.9,48.8
360,H-002-001,Kampung Paloh,N.26 Tebing Tinggi,313.3,37.2,0.0,85.4,29.3,15.9,30.0,25.4
189,H-006-004,Taman Arkid,N.32 Menglembu,196.7,59.1,14.9,95.1,37.8,56.0,69.4,37.1
433,H+000-001,Ipoh,N.27 Pasir Pinji,330.5,35.2,0.7,98.8,7.7,18.2,15.6,35.5


Saved outputs/ipoh_ageing_index_by_hex.csv


# PART 8 · WHAT THIS NOTEBOOK CANNOT MEASURE (needs the survey or a data request)
| Domain | Missing measure | How to get it |
|---|---|---|
| Money | Seniors receiving Bantuan Warga Emas | Data request to JKM, by DUN or mukim |
| Money | Main income source; ability to pay a RM1,000 surprise bill | Survey |
| Health | Self-rated health, chronic conditions, 5+ medicines | Survey (Perak benchmark from NHMS) |
| Health | Unmet need for care, and why | Survey |
| Health | Private clinics and pharmacies (OSM is incomplete) | Licensed private clinic register from JKN Perak |
| Care | Care-home places and day-care capacity | Licensed care-home register from JKM |
| Care | Help needed with daily tasks (Katz ADL, Lawton IADL) and whether they get it; living alone | Survey |
| Social | Loneliness (UCLA-3), social network size (Lubben-6), contact frequency | Survey |
| Access | Car or driver in the household, smartphone/e-hailing use, falls | Survey |
| Access | Rent burden, distance to amenities | DOSM *Living Quarters* report (one figure for the whole Ipoh City Council area) |

Suggested survey sample: **400–600 people aged 65+**, split across DUN × index band, so survey results can be attached to these maps.